# 04 V3 - ACDC avec sous-spins dans l’épaisseur et dans le plan

Cette version reprend le notebook 04, ses courants **±3 A par canal / 30 A au
total**, son filtre à **133 µs** et sa reconstruction continue. Elle utilise
**2 × 2 × 8 = 32 sous-spins par voxel parent**, dans l'ordre
`[ligne, colonne, épaisseur]`. L'image reconstruite reste à la matrice du scan.

La V3 conserve les 8 positions dans l’épaisseur et ajoute 2 × 2 positions dans le plan. Le B0 est interpolé directement depuis la carte native 128 × 128, sans passer par son interpolation sur la grille 64 × 64.

À **chaque évaluation de l'optimisation**, la phase de chaque sous-spin dépend
de son propre B0 et de ses 32 réponses ACDC. Les contributions complexes sont
sommées après ces phases. Les mêmes courants commandent tous les sous-spins ;
les contraintes de champ portent sur chaque sous-spin fiable.

La carte B0 est mesurée sur une seule coupe : sa variation suivant l'épaisseur
reste inconnue et est supposée nulle. Le profil de coupe est uniforme et la RF
idéale. L'amplitude est le proxy GRE du 04, partagé entre les sous-spins ; ce
modèle ne reconstruit pas le signal déjà perdu dans la mesure GRE.

Les fonctions communes sont dans `subspin_sampling.py` et `experiment_subspins.py`.
Les notebooks 04 et 03 restent indépendants. Le CSV est désactivé par défaut.

In [ ]:
# 1. Imports and input/output paths
%matplotlib inline
import hashlib
import json
from datetime import datetime
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "Experiment").is_dir() else Path.cwd().parent
EXPERIMENT_DIR = PROJECT_ROOT / "Experiment"
VALIDATION_DIR = PROJECT_ROOT / "MRzero_validation"
for folder in (EXPERIMENT_DIR, VALIDATION_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from prepare_acdc_inputs import read_gre, read_scan, check_reference_sequence, check_physical_geometry
from read_acdc_sequence import read_sequence
from read_acdc_b0 import read_b0_coil_images, b0_from_coil_images
from read_acdc_calibration import read_calibration
from resample_acdc_maps import resample_to_grid
from prepare_acdc_simulation_grid import prepare_simulation_reference
from run_acdc_experiment import check_calibration, current_windows
from export_acdc_waveform import export_waveform, quantize_currents
from acdc_gpu import select_device
from acdc_gpu_field import solve_field_gpu
from experiment_realistic import RealisticACDCModel
from experiment_subspins import SubspinACDCModel
from subspin_sampling import prepare_subspin_inputs
from continuous_reconstruction import ContinuousReconstruction
from measured_continuous import reconstruct_receivers, reconstruct_b0_echoes
from experiment_optimization import optimize_mrzero_image
from live_optimization import LiveOptimizationPlot

INPUT_DIR = PROJECT_ROOT / "acquisition_014"

def unique_input(pattern):
    """Find exactly one input; explain missing or ambiguous acquisitions."""
    # Baseline acquisitions are at the input root; results/ contains later scans.
    matches = sorted(INPUT_DIR.glob(pattern))
    if not matches:
        matches = sorted(path for path in INPUT_DIR.rglob(pattern)
                         if "results" not in path.relative_to(INPUT_DIR).parts)
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {pattern} under {INPUT_DIR}; found {len(matches)}. "
            "Set INPUT_DIR to the acquisition folder."
        )
    return matches[0]

FILES = {
    "reference": unique_input("*reference.dat"),
    "b0": unique_input("*b0_map.dat"),
    "scan": unique_input("*scan.dat"),
    "calibration": INPUT_DIR / "ACDC_3T.npz",
    "scan_sequence": INPUT_DIR / "sequences/gre_epi_64x64_fov200mm_nav3_rise1ms_ext1_single_trigger_first_readout_v4.seq",
    "reference_sequence": INPUT_DIR / "sequences/gre_32x32_fov200mm_tr1s.seq",
    "b0_sequence": INPUT_DIR / "sequences/b0_map.seq",
}
SESSION = datetime.now().strftime("mrzero04_v3_%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = VALIDATION_DIR / "results_experiment_realistic_v3" / SESSION
print("Input folder:", INPUT_DIR)
print("Results folder:", OUTPUT_DIR)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

## 2. Paramètres et caractéristiques documentées

| Paramètre | Valeur | Origine / interprétation |
|---|---:|---|
| Courant recommandé par canal | ±3 A | Manuscrit §3.1, p. PDF 6 ; les 5 A crête de l'OPA548 ne sont pas une consigne d'exploitation |
| Montée à 99 %, échelon 0 -> 1 A | 133 µs | §2.4.3 et Fig. 4A, pp. PDF 6-7 |
| Approximation temporelle | τ = 133 µs / ln(100) | Premier ordre équivalent ; bande passante déduite ≈5.5 kHz, sans second filtre ajouté |
| Mise à jour maximale | 125 kS/s, intervalle ≥8 µs | §3.1.1, p. PDF 6 ; aucun ADC déplacé |
| Résolution effective annoncée | ≈0.6 mA, DAC 16 bits | §2.4.3, p. PDF 6 ; pas un pas exact de quantification |
| Contraintes de courant | canal et somme des valeurs absolues | Chapitre 6 §6.5.3, p. PDF 11 |
| Exemple expérimental du manuscrit | ±2 A/canal, 10 A au total | Fig. 6, p. PDF 9 ; protocole, pas limite universelle du système |

`CURRENT_PRESET="notebook03"` conserve ±3 A/canal et 30 A au total. **Le budget
30 A est le choix confirmé par l'utilisateur le 16/10/2026, pas une spécification des PDF.** `"article_demo"` reprend ±2 A et
10 A. Les 32 canaux viennent de la calibration ; le manuscrit décrit jusqu'à
64 sorties et une démonstration avec une antenne de 31 éléments.

$$I_c(t)=u_c+(I_c(t_0)-u_c)e^{-(t-t_0)/\tau_c},\quad
\phi_{ACDC}(r,t)=-2\pi\sum_c b_c(r)\int_0^t I_c(s)\,ds.$$

Les exponentielles sont intégrées exactement. Les articles ne donnent pas les
R/L de **cette installation**, son seuil firmware de courant moyen, sa tension
de conformité ni son slew maximal garanti : ces paramètres restent `None`.
La ligne d'alimentation à 20 V n'est pas assimilée à la conformité disponible
pour la charge. Le modèle RL facultatif ne décrit pas les pertes internes.

Le filtre de 133 µs et les limites ±3 A / 30 A ont été confirmés par l'utilisateur le 16/10/2026.

In [ ]:
# Reconstruction and physical registration
INPUT_ROTATION_DEG = 180  # 180 : retourner les deux axes spatiaux ; 0 : orientation native.
REFERENCE_THRESHOLD = 0.1
NAVIGATOR_PHASE_BRANCH = 0  # 0: small odd/even mismatch; 1: alternative pi branch.
B0_SIGN = 1
GEOMETRY_CONFIRMED = True
B0_SIGN_CONFIRMED = True
T2STAR_S = np.inf  # No measured T2* map: retain the existing no-decay assumption.
B0_PARAMETERS = {
    "magnitude_fraction": 0.05,
    "max_phase_residual_rad": 0.5,
    "min_component_pixels": 10,
    "frequency_offset_hz": 0.0,
}
CALIBRATION_PARAMETERS = {
    "units": "Hz/A",
    "registration_confirmed": True,
    "patient_to_source_m": np.eye(4).tolist(),
    "source_sha256": "bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note": "Bay5 coils and alignment confirmed by the user on 2026-10-03.",
}

# Matériel : "auto", "v100", "gtx1660ti", "blackwell", "cpu".
GPU_PROFILE = "auto"
GPU_PRECISION = "auto"  # "auto" = float32 ; "float64" pour vérifier la précision.
GPU_MEMORY_FRACTION = 0.35  # Fraction de la mémoire actuellement libre pour le modèle.
SAMPLE_CHUNK = None  # None : automatique ; sinon, nombre d'ADC par bloc.
RUN_BENCHMARK = True  # Mesure un gradient après échauffement CUDA.
RUN_FORWARD_MODEL = True  # Afficher les jumeaux ; la référence reste nécessaire au solveur.
RUN_OPTIMIZATION = True
IMAGE_INITIALIZATION = "static"  # "static" ou "zero".
FIELD_LIMIT_HZ = 500.0
CURRENT_PRESET = "notebook03"  # "notebook03" ou "article_demo".
CURRENT_LIMITS = {"notebook03": (3.0, 30.0), "article_demo": (2.0, 10.0)}
MAX_CURRENT_A, TOTAL_CURRENT_LIMIT_A = CURRENT_LIMITS[CURRENT_PRESET]
MAX_ITERATIONS = 150
IMAGE_LEARNING_RATE_A = 0.01
IMAGE_PATIENCE = 40
IMAGE_TARGET_MSE = None  # Arrêt facultatif lorsque cette erreur relative est atteinte.
FIELD_SETTINGS = {"profile": GPU_PROFILE, "precision": GPU_PRECISION}

# ACDC temporal response (EPI only). Tau is in microseconds here.
CURRENT_RESPONSE_ENABLED = True
CURRENT_RESPONSE_TAU_US = 133.0 / np.log(100.0)  # 99% of a 0-to-1 A step in 133 us.
# To use measured channel values, replace the scalar above with 32 positive values
# in calibration channel order, for example: [28.9, 30.0, ...].
# Assumption: actual currents are zero before the first imaging command.
assert isinstance(CURRENT_RESPONSE_ENABLED, (bool, np.bool_)), "Use True or False."
tau_us = np.asarray(CURRENT_RESPONSE_TAU_US, dtype=float)
if tau_us.ndim == 0:
    tau_us = np.full(32, float(tau_us))
assert tau_us.shape == (32,) and np.isfinite(tau_us).all() and (tau_us > 0).all(),     "CURRENT_RESPONSE_TAU_US needs one positive scalar or 32 positive channel values."
CURRENT_RESPONSE = {"enabled": CURRENT_RESPONSE_ENABLED, "tau_s": tau_us * 1e-6}

# CSV column order and numerical current limits.
WAVEFORM_PARAMETERS = {
    "channel_map": [i for i in range(1, 33)],  # Enter 32 distinct physical output numbers in 1..64, in calibration order.
    "channel_polarity": [1] * 32,
    "max_current_A": MAX_CURRENT_A,
    "total_current_limit_A": TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s": 8e-6,  # Maximum documenté : 125 kS/s.
}
RF_DELAY_S = 0.0  # 0: times from the first RF center; enter the measured delay for trigger-relative times.
EXPORT_WAVEFORM = False

# Rendering is periodic; numeric history keeps every iteration.
# Plot every 20 iterations; set 10 for more frequent updates, or 1 for every step.
LIVE_UPDATE_EVERY = 20
# Replay identical CUDA operations without resubmitting each small kernel.
USE_CUDA_GRAPH = False  # Le helper du 03 contournerait les contraintes du 04.


# Valeurs mesurées de l'installation : scalaire ou liste en ordre calibration.
HARDWARE = {
    "min_update_interval_s": 8e-6,
    "dac_resolution_A": 0.0006,  # Informatif : ne définit pas le code DAC.
    "max_slew_A_per_s": None,
    "avg_abs_current_A": None,  # Moyenne |I| par canal sur le TR.
    "coil_R_ohm": None,
    "coil_L_H": None,
    "max_voltage_V": None,
    "max_coil_power_W": None,  # Puissance résistive moyenne par canal sur le TR.
}
RECONSTRUCTION_CHUNK = 256
assert CURRENT_RESPONSE_ENABLED, "Le 04 exige la réponse filtrée ; comparer ensuite avec forward(response_enabled=False)."
assert not USE_CUDA_GRAPH, "La projection directe est requise pour les contraintes du 04."


In [ ]:
# Quadrature au milieu de chaque sous-volume : ligne, colonne, épaisseur.
SUBSPIN_VERSION = "V3"
SUBSPIN_COUNTS = (2, 2, 8)
SUBSPIN_B0_MODE = "native"
# Le B0 mesuré n'apporte pas de gradient suivant l'épaisseur : aucun n'est inventé.
SUBSPIN_B0_Z_GRADIENT_HZ_PER_M = 0.0


In [ ]:
# Display diagnostics in the notebook without writing additional output files.
def show_figure(fig):
    plt.show()
    plt.close(fig)


def show_table(rows, headings=("Parameter", "Value")):
    lines = ["| " + " | ".join(headings) + " |", "| " + " | ".join(["---"] * len(headings)) + " |"]
    lines += ["| " + " | ".join(str(value).replace("|", "\\|") for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def rotate_input_scan(values, axes=(-2, -1)):
    """Corriger les données sur leurs deux axes spatiaux, sans changer les récepteurs."""
    return np.rot90(values, k=INPUT_ROTATION_DEG // 90, axes=axes).copy()


def rotate_scan_diagnostics(diagnostics):
    # [phase, récepteur, lecture] : le récepteur reste sur le même canal.
    for key in ("coil_images", "uncorrected_coil_images"):
        if key in diagnostics:
            diagnostics[key] = rotate_input_scan(diagnostics[key], axes=(0, 2))
    if "uncorrected_image" in diagnostics:
        diagnostics["uncorrected_image"] = rotate_input_scan(diagnostics["uncorrected_image"])
    # Le k-space et les navigateurs restent dans leur ordre d'acquisition.


def image_panel(ax, values, title, cmap="gray", **kwargs):
    artist = ax.imshow(values, cmap=cmap, interpolation="nearest", **kwargs)
    ax.set_title(title)
    ax.set_xlabel("Array column")
    ax.set_ylabel("Array row")
    return artist


assert INPUT_ROTATION_DEG in (0, 180), "INPUT_ROTATION_DEG doit valoir 0 ou 180."
print(f"Rotation des données d'entrée : {INPUT_ROTATION_DEG}° (avant ROI, B0 et optimisation).")
for name, path in FILES.items():
    assert path.is_file(), f"Missing {name}: {path}"
assert not isinstance(B0_SIGN, bool) and B0_SIGN in (-1, 1), "B0_SIGN must be +1 or -1."
assert 0 <= REFERENCE_THRESHOLD < 1, "REFERENCE_THRESHOLD must be in [0, 1)."
assert np.isscalar(T2STAR_S) and T2STAR_S > 0, "T2STAR_S must be positive; infinity is allowed."
show_table([(name, path.name) for name, path in FILES.items()])

In [ ]:
def json_value(value):
    """Convertir les tableaux pour les rapports JSON."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)

source_pdfs = {"controller": PROJECT_ROOT / "main_revised_small.pdf",
               "chapter6": PROJECT_ROOT / "UHF_Neuro_MRI_book_CH6 (2).pdf"}
paper_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest()
                for name, path in source_pdfs.items() if path.is_file()}
show_table([("Profil courant", CURRENT_PRESET), ("Limite canal [A]", MAX_CURRENT_A),
            ("Somme |I| maximale [A]", TOTAL_CURRENT_LIMIT_A),
            ("Tau [µs]", [float(tau_us.min()), float(tau_us.max())])])


## 3. Sequence trajectory and acquisition timing

The trajectory is derived from the actual RF/ADC/gradient events. The `Matrix`, `FOV`,
`TE` and `TR` definitions are checked against these events. Cartesian order is used
for reconstruction, while `line_index` preserves chronological current order.
For spoiled GRE every line starts a new transverse phase history; EPI retains one RF history.

In [ ]:
# Read the scan and reference sequences independently.
acquisition, timing = read_sequence(FILES["scan_sequence"])
assert not CURRENT_RESPONSE_ENABLED or timing["sequence_kind"] == "epi", "Disable the temporal response for a GRE scan; this model currently supports EPI."
reference_acquisition, reference_timing = read_sequence(FILES["reference_sequence"])
show_table([
    ("Scan kind", timing["sequence_kind"]),
    ("Scan matrix [Nx, Ny]", timing["matrix_size"]),
    ("Scan TE / TR [ms]", np.array([acquisition["TE_s"], acquisition["TR_s"]]) * 1000),
    ("ADC dwell [us]", acquisition["dwell_s"] * 1e6),
    ("Reference matrix [Nx, Ny]", reference_timing["matrix_size"]),
    ("Reference TE / TR [ms]", np.array([reference_acquisition["TE_s"], reference_acquisition["TR_s"]]) * 1000),
])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
plot = axes[0].scatter(acquisition["kx_mInv"], acquisition["ky_mInv"],
                       c=acquisition["time_s"] * 1000, s=6, cmap="viridis")
axes[0].set(xlabel="kx [cycles/m]", ylabel="ky [cycles/m]", title="ADC trajectory, colored by time")
fig.colorbar(plot, ax=axes[0], label="Time from first RF center [ms]")
plot = image_panel(axes[1], acquisition["line_index"], "Chronological line index", cmap="viridis")
fig.colorbar(plot, ax=axes[1])
starts, ends = current_windows(timing)
for line, (start, end) in enumerate(zip(starts, ends)):
    axes[2].plot([start * 1000, end * 1000], [line + 1, line + 1], color="tab:blue")
for center in np.unique(acquisition["rf_time_s"]):
    axes[2].axvline(center * 1000, color="tab:red", alpha=0.3)
axes[2].set(xlabel="Time from first RF [ms]", ylabel="Current state", title="Current windows and RF centers")
show_figure(fig)

### 3a. Reconstruction continue et audit de la grille

Le 03 utilise déjà les coordonnées continues pour simuler le signal et ne
présente pas le défaut d'arrondi/suppression du 02. Ici, la reconstruction et
son adjoint utilisent également ces coordonnées pour les simulations et les
données mesurées, par récepteur avant RSS, y compris les échos B0.

$$[R s]_p=\frac{1}{N_{ADC}}\sum_j s_j e^{2\pi i(k_{x,j}x_p+k_{y,j}y_p)}.$$

Sur une grille cartésienne exacte, R est l'IFFT centrée, avec une rampe de phase
si la grille a un décalage constant. Hors grille, c'est une **rétroprojection de
Fourier normalisée**, pas une inversion itérative. Tous les ADC sont conservés.
Les trajectoires sont calculées à partir du `.seq` : retards de gradients et
courants de Foucault réels ne sont pas mesurés ici.

In [ ]:
scan_reconstruction = ContinuousReconstruction(acquisition, "cpu", torch.float64,
                                              sample_chunk=RECONSTRUCTION_CHUNK)
show_table(list(scan_reconstruction.report.items()), ("Reconstruction EPI", "Valeur"))

# Produit scalaire indépendant : <R a, b> = <a, R^H b>.
rng = np.random.default_rng(4)
shape = acquisition["kx_mInv"].shape
a = torch.as_tensor(rng.normal(size=shape) + 1j * rng.normal(size=shape))
b = torch.as_tensor(rng.normal(size=shape) + 1j * rng.normal(size=shape))
left = torch.vdot(scan_reconstruction.image(a).ravel(), b.ravel())
right = torch.vdot(a.ravel(), scan_reconstruction.adjoint(b).ravel())
adjoint_error = float(abs(left - right) / max(float(abs(left)), 1e-12))
assert adjoint_error < 1e-10
print("Erreur relative du test d'adjoint :", adjoint_error)


## 4. Reconstruct the reference and verify its physical coordinates

The reference is a **mono-echo Cartesian 2D GRE/FLASH Twix** with one RF excitation
per acquired line. Its own `reference_gre.seq` supplies the expected matrix, FOV,
TE, TR, ADC dwell, readout polarity and acquisition order. These are checked
against the raw data before defining the ROI; no image size is entered manually.

Reorder the acquired k-space lines, apply a centered inverse Fourier transform to
each receive coil, then combine magnitudes with root-sum-of-squares (RSS).
The array axes are `[phase row, readout column]`. Physical coordinates use
`X = readout` and `Y = phase`, in metres. The displayed pixel coordinates also
allow you to inspect the reference geometry before resampling the measured maps.

The RSS magnitude is an effective transverse-amplitude proxy, not a proton-density
or T1 map. The next cell defines the ROI from this reconstructed reference.

Les images reconstruites sont tournées selon `INPUT_ROTATION_DEG` avant la définition de la ROI.


In [ ]:
# Reconstruct the raw reference and validate it against its own GRE sequence.
gre, reference_diagnostics = read_gre(
    FILES["reference"], reference_acquisition, return_intermediates=True)
check_reference_sequence(gre, reference_acquisition)
gre["magnitude"], reference_reconstruction_report = reconstruct_receivers(
    reference_diagnostics, reference_acquisition)
gre["magnitude"] = rotate_input_scan(gre["magnitude"])
rotate_scan_diagnostics(reference_diagnostics)
reference_magnitude = np.asarray(gre["magnitude"], float)
reference_metadata = gre["twix_metadata"]
X_m, Y_m = np.asarray(gre["X_m"]), np.asarray(gre["Y_m"])
show_table([
    ("Reference matrix [phase rows, readout columns]", reference_magnitude.shape),
    ("Receive coils", reference_metadata["receiver_count"]),
    ("TE / TR [ms]", np.array([reference_metadata["TE_s"], reference_metadata["TR_s"]]) * 1000),
    ("ADC dwell [us]", reference_metadata["dwell_s"] * 1e6),
    ("Readout / phase FOV [mm]", np.asarray(reference_metadata["FOV_m"]) * 1000),
    ("Slice center [patient mm]", reference_metadata["slice_position_mm"]),
])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(reference_diagnostics["kspace"], axis=1)), "Reference: log k-space RSS")
image_panel(axes[1], np.abs(reference_diagnostics["coil_images"][:, 0, :]), "Reference: receiver 1 magnitude")
image_panel(axes[2], reference_magnitude, "Reference: combined RSS magnitude")
show_figure(fig)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
plot = image_panel(axes[0], np.angle(reference_diagnostics["coil_images"][:, 0, :]),
                   "Reference: receiver 1 phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
fig.colorbar(plot, ax=axes[0])
axes[1].scatter(X_m * 1000, Y_m * 1000, c=reference_magnitude, s=8, cmap="gray")
axes[1].set(xlabel="X: readout [mm]", ylabel="Y: phase [mm]", title="Physical pixel coordinates", aspect="equal")
show_figure(fig)


### 4a. Match the object sampling to the scan

The Python/PyTorch forward model sums point sources at the object pixels. A 32×32
object sampled directly into 64×64 k-space produces a dotted comb, even at
zero B0, and biases the image optimization. Before selecting the ROI, refine
the reference grid to at least the scan's physical pixel spacing on both axes.
For equal 200 mm FOVs, a 32×32 reference and 64×64 EPI use a 64×64 simulation grid.
Different FOVs are handled by physical spacing, rather than matrix size alone.

Linear interpolation retains the reference's physical FOV and Fourier origin.
It does not add measured spatial resolution. Samples outside reference coverage
are excluded. B0 and all 32 ACDC responses are then interpolated directly from
their measured source grids onto this simulation grid, with their validity masks.
Patient position and pixel spacing are updated together for ACDC placement.
The native GRE reconstruction remains available as `native_reference_magnitude`
and `gre`; the following ROI, fits and comparisons use the interpolated reference.
After changing an acquisition, rerun from the top, including all optimizations.


In [ ]:
# Keep the native measured reference; use a sufficiently fine object for Python/PyTorch.
native_reference_magnitude = np.asarray(gre["magnitude"], float).copy()
simulation_reference, simulation_grid_report = prepare_simulation_reference(gre, acquisition)
reference_magnitude = simulation_reference["magnitude"]
X_m, Y_m = simulation_reference["X_m"], simulation_reference["Y_m"]
reference_grid_valid = simulation_reference["valid_mask"]
show_table([
    ("Native measured GRE [phase rows, readout columns]", native_reference_magnitude.shape),
    ("Scan [phase rows, readout columns]", np.asarray(acquisition["kx_mInv"]).shape),
    ("Simulation object [rows, columns]", reference_magnitude.shape),
    ("Native / simulation spacing [row, column mm]", f'{simulation_grid_report["native_spacing_mm"]} / {simulation_grid_report["simulation_spacing_mm"]}'),
    ("Reference interpolation needed", simulation_grid_report["refined"]),
    ("Simulation pixels covered by the measured reference", f"{int(reference_grid_valid.sum())}/{reference_grid_valid.size}"),
])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
reference_display_scale = native_reference_magnitude.max()
image_panel(axes[0], native_reference_magnitude, "Native measured GRE", vmin=0, vmax=reference_display_scale)
image_panel(axes[1], reference_magnitude, "Reference on simulation grid", vmin=0, vmax=reference_display_scale)
show_figure(fig)


### 4b. Define and inspect the ROI

The default selects all reference pixels above `REFERENCE_THRESHOLD * maximum_magnitude`.
The threshold is set in step 2. Optional X/Y limits select a rectangle in physical
millimetres; keep both limits at `None` to select the whole thresholded object.

Change these limits and rerun this cell and the following cells. Green pixels in the
overlay are the requested ROI. The binary mask and excluded object pixels show exactly
what was selected. The computed mask stays in memory and is included in the internal Python/PyTorch input.
No mask is read from the input folder. Step 8 displays B0 coverage. In step 9,
TRIM_TO_COMMON_VALID_SUPPORT explicitly controls removal of pixels missing valid B0
or ACDC calibration (True by default, as in v3). The removed pixels and final ROI are shown.


In [ ]:
# Optional physical rectangle limits; None means no restriction on that axis.
ROI_X_LIMITS_MM = None  # Example: (-40.0, 40.0), X = readout.
ROI_Y_LIMITS_MM = None  # Example: (-40.0, 40.0), Y = phase.

threshold_support = reference_grid_valid & (reference_magnitude > REFERENCE_THRESHOLD * reference_magnitude.max())
support = threshold_support.copy()
for axis_name, limits, coordinates in (
    ("X", ROI_X_LIMITS_MM, X_m), ("Y", ROI_Y_LIMITS_MM, Y_m)
):
    if limits is not None:
        bounds = np.asarray(limits, dtype=float)
        assert bounds.shape == (2,) and np.isfinite(bounds).all() and bounds[0] <= bounds[1], f"{axis_name} limits need two finite increasing values in mm."
        support &= (coordinates * 1000 >= bounds[0]) & (coordinates * 1000 <= bounds[1])

support_before_validity_filter = support.copy()
excluded_object_pixels = threshold_support & ~support
excluded_signal_fraction = reference_magnitude[~support].sum() / reference_magnitude.sum()
fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout="constrained")
image_panel(axes[0, 0], reference_magnitude, "GRE reference magnitude")
image_panel(axes[0, 1], support, f"Computed ROI: {int(support.sum())} pixels", vmin=0, vmax=1)
image_panel(axes[1, 0], reference_magnitude, "GRE overlay: green = selected ROI")
axes[1, 0].imshow(np.ma.masked_where(~support, support), cmap="Greens", vmin=0, vmax=1, alpha=0.45, interpolation="nearest")
if support.any() and not support.all():
    axes[1, 0].contour(support, levels=[0.5], colors=["lime"], linewidths=0.8)
image_panel(axes[1, 1], excluded_object_pixels, "Thresholded object pixels outside ROI", cmap="Reds", vmin=0, vmax=1)
show_figure(fig)
show_table([
    ("Reference magnitude threshold", REFERENCE_THRESHOLD),
    ("X / Y limits [mm]", f"{ROI_X_LIMITS_MM} / {ROI_Y_LIMITS_MM}"),
    ("Thresholded object pixels", int(threshold_support.sum())),
    ("Selected ROI pixels", int(support.sum())),
    ("Reference magnitude excluded", f"{excluded_signal_fraction:.1%}"),
])
assert support.any(), "The computed ROI is empty: inspect the threshold and physical limits above."
ROI_PARAMETERS = {
    "method": "reference magnitude threshold with optional physical X/Y limits",
    "reference_threshold": REFERENCE_THRESHOLD,
    "x_limits_mm": [] if ROI_X_LIMITS_MM is None else list(ROI_X_LIMITS_MM),
    "y_limits_mm": [] if ROI_Y_LIMITS_MM is None else list(ROI_Y_LIMITS_MM),
}
print("ROI defined in memory; inspect the selection figure before continuing.")

## 5. Scan k-space, navigator correction and RSS reconstruction

The raw scan is reconstructed independently of the reference. Negative readouts
are reversed using `REFLECT`; a centered inverse FFT and RSS combine the receivers.
Before/after magnitude images share one intensity scale.

For three equally spaced `+,-,+` navigators, estimate the phase differences
`navigator 2 - navigator 1` and `navigator 2 - navigator 3` separately. Their
average removes approximately linear B0 phase evolution between the echoes.
Do **not** average the complex positive echoes: they can cancel almost completely.
Each receiver's affine phase fit uses only its strongest contiguous region with
reliable signal in all three navigators; noisy background is not phase-unwrapped.

The correction has a **pi branch ambiguity**. `NAVIGATOR_PHASE_BRANCH = 0` assumes
a small odd/even phase mismatch at the strongest signal sample; `1` selects the
alternative branch. Verify this convention experimentally. Neither the GRE nor
image registration is used to choose the branch. Constant B0 during the three
navigators is assumed. The sequence check rejects asymmetric navigator spacing.

Inspect the selected fit pixels, phase residuals and cancelled-average diagnostic
below. Receivers with fewer than four contiguous reliable pixels keep zero
correction and are reported explicitly. This step preserves the physical EPI B0
displacement and distortion; it does not shift the image to match the reference.

A good navigator phase fit does not guarantee an artifact-free image. A uniform
B0 offset also displaces the EPI along the phase axis and can wrap the object
across the image boundary. Section **8b** checks this separately using the
measured field map. A weaker half-FOV ghost may remain after both operations.


In [ ]:
# Inspect reconstruction before and after navigator correction.
# Reload and rebind the reader without recreating the Python/PyTorch workspace.
import importlib
import prepare_acdc_inputs
read_scan = importlib.reload(prepare_acdc_inputs).read_scan
print("Reconstruction reader:", Path(prepare_acdc_inputs.__file__).resolve())

scan_image, scan_metadata, scan_diagnostics = read_scan(
    FILES["scan"], acquisition, return_intermediates=True,
    navigator_phase_branch=NAVIGATOR_PHASE_BRANCH)
scan_image, measured_scan_reconstruction_report = reconstruct_receivers(scan_diagnostics, acquisition)
scan_image = rotate_input_scan(scan_image)
rotate_scan_diagnostics(scan_diagnostics)
check_physical_geometry(gre, scan_metadata, acquisition, support, X_m, Y_m, dicom_input=False)
for key in ("TE_s", "TR_s", "dwell_s"):
    tolerance = 1e-9 if key == "dwell_s" else 1e-6
    assert abs(scan_metadata[key] - acquisition[key]) <= tolerance, f"Scan header disagrees on {key}."
assert scan_metadata["navigator_count"] == timing["navigator_count"], "Navigator count mismatch."
assert np.array_equal(scan_metadata["polarity"], acquisition["polarity"]), "Readout polarity mismatch."
assert scan_metadata["acquired_phase_rows"] == np.argsort(acquisition["line_index"][:, 0]).tolist(), "Acquisition order mismatch."
nav_scale = max(scan_diagnostics["uncorrected_image"].max(), scan_image.max())
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(scan_diagnostics["kspace"], axis=1)), "Raw reordered k-space: log RSS")
image_panel(axes[1], np.log1p(np.linalg.norm(scan_diagnostics["corrected_kspace"], axis=1)), "Corrected k-space: log RSS")
image_panel(axes[2], scan_diagnostics["uncorrected_image"], "RSS before navigator correction", vmin=0, vmax=nav_scale)
image_panel(axes[3], scan_image, "RSS after navigator correction", vmin=0, vmax=nav_scale)
show_figure(fig)
if scan_metadata["navigator_count"]:
    nav = scan_diagnostics["navigator_hybrid"]
    nav_fit = scan_diagnostics["navigator_fit"]
    fitted = nav_fit["fitted_channels"]
    fit_pixels = nav_fit["fit_mask"].sum(axis=1)
    skipped_receivers = (np.flatnonzero(~fitted) + 1).tolist()
    show_table([
        ("Navigator phase branch", NAVIGATOR_PHASE_BRANCH),
        ("Receivers with a valid phase fit", f"{int(fitted.sum())}/{len(fitted)}"),
        ("Receivers left uncorrected", skipped_receivers),
    ])
    if fitted.any():
        retention = nav_fit["outer_average_retention"][nav_fit["fit_mask"]]
        show_table([
            ("Selected fit pixels per receiver [min, max]", [int(fit_pixels[fitted].min()), int(fit_pixels[fitted].max())]),
            ("Median phase fit residual [rad]", float(np.median(nav_fit["fit_residual_rad"][fitted]))),
            ("Median amplitude retained by complex outer average", f"{np.median(retention):.1%}"),
        ])
        receiver = int(np.argmax(np.where(fitted, np.abs(nav).sum(axis=(0, 2)), -1)))
        columns = np.arange(scan_image.shape[1])
        selected = nav_fit["fit_mask"][receiver]
        fig, axes = plt.subplots(1, 3, figsize=(17, 4), layout="constrained")
        for index in range(3):
            axes[0].plot(columns, np.abs(nav[index, receiver]), label=f"Navigator {index + 1}")
        axes[0].plot(columns, np.abs((nav[0, receiver] + nav[2, receiver]) / 2),
                     "k--", label="Complex outer average (unused)")
        axes[0].axvspan(columns[selected][0], columns[selected][-1], color="green", alpha=0.12, label="Fit region")
        axes[0].set(xlabel="Readout column", ylabel="Magnitude", title=f"Navigator signal: receiver {receiver + 1}")
        axes[0].legend(fontsize=8)
        for pair, label in enumerate(("Phase 2 - 1", "Phase 2 - 3")):
            axes[1].plot(columns[selected], nav_fit["pair_phase_rad"][pair, receiver, selected], ".-", label=label)
        axes[1].plot(columns[selected], nav_fit["phase_estimate_rad"][receiver, selected], "k.", label="Mean phase, selected branch")
        axes[1].plot(columns[selected], scan_diagnostics["navigator_phase_correction_rad"][receiver, selected], "k-", label="Affine fit")
        axes[1].set(xlabel="Readout column", ylabel="Phase [rad]", title="Separate pair phases and fitted correction")
        axes[1].legend(fontsize=8)
        image_panel(axes[2], nav_fit["fit_mask"], "Reliable contiguous pixels used for each fit", vmin=0, vmax=1)
        axes[2].set(xlabel="Readout column", ylabel="Receiver index")
        show_figure(fig)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    image_panel(axes[0], np.angle(nav[:, 0]), "Navigator phases, receiver 1", cmap="twilight", vmin=-np.pi, vmax=np.pi)
    plot = image_panel(axes[1], scan_diagnostics["navigator_phase_correction_rad"], "Fitted phase correction [rad]", cmap="RdBu_r")
    axes[1].set(xlabel="Readout column", ylabel="Receiver index")
    fig.colorbar(plot, ax=axes[1])
    show_figure(fig)
print("Scan matrix:", scan_image.shape, "; receiver count:", scan_metadata["receiver_count"])


## 6. Multi-echo B0: complex images before field estimation

B0 requires phase information from at least two equally spaced monopolar echoes.
First inspect each echo's k-space, RSS magnitude and one receiver's phase. This cell
runs before masking or phase unwrapping, so acquisition/reconstruction problems remain visible.
Les deux axes spatiaux des échos complexes sont tournés selon `INPUT_ROTATION_DEG` avant le calcul du champ.


In [ ]:
# Read complex echoes without yet estimating a field.
b0_raw = read_b0_coil_images(FILES["b0"], FILES["b0_sequence"])
# Reconstruire aux coordonnées ADC, puis appliquer la rotation.
b0_reconstruction_reports = reconstruct_b0_echoes(b0_raw, FILES["b0_sequence"])
b0_raw["coil_images"] = rotate_input_scan(b0_raw["coil_images"])
coils = b0_raw["coil_images"]  # [echo, receiver, phase row, readout column]
te_s = np.asarray(b0_raw["metadata"]["sequence_info"]["TE_s"])
assert abs(b0_raw["metadata"]["slice_position_mm"][2] - gre["position_mm"][2]) <= 1e-3, "B0 and reference slice planes differ."
assert np.isclose(abs(np.dot(b0_raw["metadata"]["slice_quaternion_wxyz"], scan_metadata["slice_quaternion_wxyz"])), 1, atol=1e-8), "B0 and scan orientations differ."
show_table([("B0 matrix", coils.shape[2:]), ("Echoes", coils.shape[0]),
            ("Receive coils", coils.shape[1]), ("Echo times [ms]", te_s * 1000)])
ne = coils.shape[0]
fig, axes = plt.subplots(3, ne, figsize=(5 * ne, 11), layout="constrained", squeeze=False)
for echo in range(ne):
    image_panel(axes[0, echo], np.log1p(np.linalg.norm(b0_raw["kspace"][echo], axis=0)), f"Echo {echo + 1}: log k-space RSS")
    image_panel(axes[1, echo], np.linalg.norm(coils[echo], axis=0), f"Echo {echo + 1}: magnitude RSS")
    image_panel(axes[2, echo], np.angle(coils[echo, 0]), f"Echo {echo + 1}: receiver 1 phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
show_figure(fig)

## 7. Phase differences, reliability masks and measured field

For each receiver, `echo[n+1] * conj(echo[n])` cancels its static receive phase. The
products are summed across receivers and echo pairs. The field convention is
`signal(TE) = amplitude * exp(-2π i B0 TE)`, followed by the explicit scanner sign choice.

Reliability requires sufficient magnitude, consistent adjacent-echo phase increments,
and components large enough for spatial phase unwrapping. No B0 value is extrapolated.
Each component is anchored at its strongest pixel in the principal frequency band;
this assumption does not measure the absolute frequency branch.

In [ ]:
# Show phase increments before spatial unwrapping.
echo_products = np.sum(coils[1:] * coils[:-1].conj(), axis=1)
wrapped_increment = np.angle(echo_products.sum(axis=0))
fig, axes = plt.subplots(1, ne, figsize=(5 * ne, 4), layout="constrained", squeeze=False)
for pair in range(ne - 1):
    image_panel(axes[0, pair], np.angle(echo_products[pair]), f"Echo pair {pair + 1}: wrapped increment", cmap="twilight", vmin=-np.pi, vmax=np.pi)
image_panel(axes[0, -1], wrapped_increment, "Combined wrapped increment [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
show_figure(fig)

In [ ]:
# Estimate the field with the same tested reconstruction as the pipeline.
b0_result = b0_from_coil_images(coils, te_s, return_intermediates=True, **B0_PARAMETERS)
b0_result["b0Hz"] *= B0_SIGN
b0_result["X_m"], b0_result["Y_m"] = b0_raw["X_m"], b0_raw["Y_m"]
b0_details = b0_result["intermediates"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout="constrained")
image_panel(axes[0, 0], b0_details["magnitude_mask"], "Magnitude threshold passed")
image_panel(axes[0, 1], b0_details["phase_residual_mask"], "Echo consistency passed")
image_panel(axes[0, 2], b0_result["mask"], "Final reliability mask")
plot = image_panel(axes[1, 0], b0_result["phase_residual_rad"], "Echo increment disagreement [rad]", cmap="magma", vmin=0, vmax=np.pi)
fig.colorbar(plot, ax=axes[1, 0])
plot = image_panel(axes[1, 1], b0_details["component_labels"], "Connected field components", cmap="tab20")
plot = image_panel(axes[1, 2], b0_result["b0Hz"], "Measured B0 [Hz], masked", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[1, 2])
show_figure(fig)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
image_panel(axes[0], b0_details["quality_before_component_filter"], "Magnitude and phase checks passed")
image_panel(axes[1], b0_result["mask"], "Large reliable components retained")
image_panel(axes[2], b0_details["quality_before_component_filter"] & ~b0_result["mask"], "Small components removed")
show_figure(fig)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.hist(b0_result["b0Hz"][b0_result["mask"]], bins=60)
ax.set(xlabel="Measured B0 [Hz]", ylabel="Reliable pixels", title="Masked field distribution")
show_figure(fig)
show_table([("Reliable native-grid pixels", int(b0_result["mask"].sum())),
            ("Frequency band at component anchors [Hz]", b0_result["metadata"]["principal_frequency_band_hz"]),
            ("B0 sign applied", B0_SIGN)])

## 8. Resample measured maps onto the simulation grid

The B0 and simulation grids are independent. Linear interpolation accepts a pixel only
when every contributing source measurement is valid. Missing measurements are displayed.
The requested ROI is retained at this stage. In section 9,
`TRIM_TO_COMMON_VALID_SUPPORT = True` removes pixels lacking valid B0 or calibration
and shows the final ROI; with False, incomplete coverage blocks simulation.


In [ ]:
# Interpolate without filling holes or extending the measured field.
b0_on_reference, b0_valid = resample_to_grid(
    b0_result["b0Hz"], b0_result["X_m"], b0_result["Y_m"], b0_result["mask"], X_m, Y_m)
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], threshold_support, "Full reference support")
image_panel(axes[1], b0_valid, "Valid B0 on reference grid")
image_panel(axes[2], support, "Selected simulation support")
plot = image_panel(axes[3], np.where(b0_valid, b0_on_reference, np.nan), "Measured B0 on reference [Hz]", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[3])
show_figure(fig)
missing_b0_pixels = support & ~b0_valid
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image_panel(ax, reference_magnitude, "Requested ROI: red = missing reliable B0")
ax.imshow(np.ma.masked_where(~missing_b0_pixels, missing_b0_pixels), cmap="Reds", vmin=0, vmax=1, alpha=0.65, interpolation="nearest")
if support.any() and not support.all():
    ax.contour(support, levels=[0.5], colors=["lime"], linewidths=0.8)
show_figure(fig)
show_table([("Full support covered by B0", f"{int((threshold_support & b0_valid).sum())}/{int(threshold_support.sum())}"),
            ("Selected ROI covered by B0", f"{int((support & b0_valid).sum())}/{int(support.sum())}"),
            ("Selected ROI pixels missing reliable B0", int(missing_b0_pixels.sum())),
            ("Reference signal excluded by ROI", f"{excluded_signal_fraction:.1%}")])
assert GEOMETRY_CONFIRMED and B0_SIGN_CONFIRMED, "Confirm the physical geometry and scanner sign in the parameter cell."


## 9. Cartes ACDC et préparation du simulateur MRzero

On conserve les cartes mesurées, les coordonnées, la ROI et les unités du
notebook source. L'aimantation de base est maintenant simulée par MRzero,
avec les mêmes temps ADC, phases RF/ADC et trajectoires du fichier Pulseq.
Les deux simulations natives (B0 mesuré et B0 nul) sont préparées une fois.
Les blocs de calcul règlent l'utilisation mémoire sans retirer de voxels ni d'ADC.

In [ ]:
# Sample the measured calibration directly on the simulation grid.
geometry = {key: simulation_reference[key] for key in ("X_m", "Y_m", "position_mm", "spacing_mm", "orientation", "slice_thickness_mm")}
calibration = read_calibration(FILES["calibration"], {"geometry": geometry}, CALIBRATION_PARAMETERS)
coil_maps = calibration["acdc_maps"]
calibration_valid = calibration["spatial_valid_mask"]
fig, axes = plt.subplots(4, 8, figsize=(19, 10), layout="constrained")
scale = np.nanpercentile(np.abs(coil_maps), 99)
for channel, ax in enumerate(axes.ravel()):
    ax.imshow(coil_maps[..., channel], cmap="RdBu_r", vmin=-scale, vmax=scale, interpolation="nearest")
    ax.set_title(f"Channel {channel + 1}")
    ax.set_axis_off()
fig.suptitle(f"Measured ACDC responses on reference grid [Hz/A], common scale ±{scale:.1f}")
show_figure(fig)
print("Calibration shape:", coil_maps.shape, "; units:", calibration["units"])

In [ ]:
TRIM_TO_COMMON_VALID_SUPPORT = True
assert support_before_validity_filter.shape == calibration_valid.shape == b0_valid.shape, "ROI, ACDC, and B0 masks must share the simulation grid."
valid_acdc = calibration_valid & np.isfinite(coil_maps).all(axis=-1)
valid_b0 = b0_valid & np.isfinite(b0_on_reference)
common_valid = valid_acdc & valid_b0
acdc_invalid_pixels = support_before_validity_filter & ~valid_acdc
b0_invalid_pixels = support_before_validity_filter & ~valid_b0
invalid_for_either = acdc_invalid_pixels | b0_invalid_pixels
invalid_for_both = acdc_invalid_pixels & b0_invalid_pixels
removed_pixels = invalid_for_either.copy() if TRIM_TO_COMMON_VALID_SUPPORT else np.zeros_like(support_before_validity_filter, dtype=bool)
support = support_before_validity_filter & common_valid if TRIM_TO_COMMON_VALID_SUPPORT else support_before_validity_filter.copy()
excluded_object_pixels = threshold_support & ~support
excluded_signal_fraction = reference_magnitude[~support].sum() / reference_magnitude.sum()
ROI_PARAMETERS.update({
    "common_validity_filter_enabled": bool(TRIM_TO_COMMON_VALID_SUPPORT),
    "selected_pixels_invalid_for_acdc_calibration": int(acdc_invalid_pixels.sum()),
    "selected_pixels_invalid_for_b0": int(b0_invalid_pixels.sum()),
    "selected_pixels_invalid_for_both": int(invalid_for_both.sum()),
    "pixels_removed_for_invalid_acdc_or_b0": int(removed_pixels.sum()),
    "selected_pixels_after_common_validity_filter": int(support.sum()),
})
fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="constrained")
image_panel(axes[0], support_before_validity_filter, f"Original support: {int(support_before_validity_filter.sum())} pixels", vmin=0, vmax=1)
image_panel(axes[1], valid_acdc, "Valid ACDC responses", vmin=0, vmax=1)
image_panel(axes[2], valid_b0, "Finite, valid B0", vmin=0, vmax=1)
image_panel(axes[3], support, f"Common-valid support: {int(support.sum())} pixels", vmin=0, vmax=1)
show_figure(fig)
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image_panel(ax, reference_magnitude, f"Pixels removed for invalid ACDC or B0: {int(removed_pixels.sum())}")
removed_rows, removed_columns = np.where(removed_pixels)
ax.scatter(removed_columns, removed_rows, marker="s", s=28, c="red")
show_figure(fig)
show_table([("Common-validity filter enabled", TRIM_TO_COMMON_VALID_SUPPORT),
            ("Pixels invalid for ACDC", int(acdc_invalid_pixels.sum())),
            ("Pixels invalid for B0", int(b0_invalid_pixels.sum())),
            ("Pixels invalid for both", int(invalid_for_both.sum())),
            ("Pixels removed (unique total)", int(removed_pixels.sum())),
            ("Selected pixels remaining", int(support.sum())),
            ("Reference signal excluded", f"{excluded_signal_fraction:.1%}")])
assert support.any(), "ACDC/B0 validity filtering removed the entire selected support."
assert np.all(common_valid[support]), "Selected support contains an invalid ACDC or B0 pixel; enable TRIM_TO_COMMON_VALID_SUPPORT."

### 9a. Préparer les sous-spins avant toute simulation MRzero

L'épaisseur est celle du **scan EPI**. Les positions dans le plan sont des
milieux de sous-voxels dans le repère physique hérité du 04. Le B0 natif est
réutilisé avec sa rotation et son signe déjà appliqués : aucune seconde
rotation n'est effectuée. Les cartes ACDC sont interpolées dans leur volume 3D.

Un parent est retiré si un seul de ses sous-spins manque de B0 ou de calibration
fiable. Les poids des sous-spins d'un parent conservé somment à 1. L'amplitude
n'est pas renormalisée après ce retrait ; le signal exclu est indiqué.

In [ ]:
# Prepare the physical inputs, then native MRzero magnetization once.
scale = reference_magnitude[support].max()
m0_proxy = np.where(support, reference_magnitude / scale, 0.0)
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in FILES.items()}
experiment = {
    "images": {"gre_reference": reference_magnitude, "scan": scan_image, "perturbed": scan_image},
    "field": {"b0Hz": np.where(support, b0_on_reference, 0.0),
              "measured_b0Hz": np.where(b0_valid, b0_on_reference, np.nan),
              "mask": b0_valid, "simulation_mask": support},
    "geometry": geometry, "acquisition": acquisition, "m0_proxy": m0_proxy,
    "relaxation": {"T2star_s": T2STAR_S},
    "current_response": CURRENT_RESPONSE,
    "assumptions": np.array([
        "GRE magnitude is a transverse-amplitude proxy; reference/scan contrast is not calibrated.",
        "Only the explicitly chosen ROI is simulated; every ROI field value is measured.",
        "Native MRzero point spins; uniform receive sensitivity; ideal GRE spoiling.",
        "ACDC: first-order temporal response when enabled; zero initial current; experimental validation required.",
        "Component frequency branches and scanner B0 sign need independent verification.",
    ], dtype=object),
    "provenance": {"source_sha256": source_hashes, "B0_sign": B0_SIGN,
                   "input_rotation_deg": INPUT_ROTATION_DEG,
                   "input_rotation_convention": "Reconstructed GRE/scan/B0 data rotated on fixed physical grids; calibration unchanged.",
                   "GRE_amplitude_normalization": float(scale), "roi_parameters": ROI_PARAMETERS,
                   "simulation_grid": simulation_grid_report,
                   "navigator_phase_branch": NAVIGATOR_PHASE_BRANCH,
                   "selected_roi_sha256": hashlib.sha256(np.ascontiguousarray(support, dtype=np.uint8).tobytes()).hexdigest()},
    "optimization_ready": True,
}
# La coupe à intégrer est celle de l'EPI ; le GRE fournit seulement l'amplitude.
geometry = dict(geometry, slice_thickness_mm=float(scan_metadata["slice_thickness_mm"]))
experiment["geometry"] = geometry
check_calibration(calibration, experiment)


In [ ]:
# La V3 interroge b0_result (grille native), jamais b0_on_reference.
subspin_data = prepare_subspin_inputs(
    experiment, FILES["calibration"], CALIBRATION_PARAMETERS,
    b0_native=b0_result, counts=SUBSPIN_COUNTS, b0_mode=SUBSPIN_B0_MODE,
    z_gradient_hz_per_m=SUBSPIN_B0_Z_GRADIENT_HZ_PER_M,
)
subspin_report = subspin_data["report"]
support_before_subspins = support.copy()
support = subspin_data["parent_valid_mask"].copy()
m0_proxy = np.where(support, m0_proxy, 0.0)
experiment["m0_proxy"] = m0_proxy
experiment["field"] = dict(experiment["field"], simulation_mask=support,
                           b0Hz=np.where(support, b0_on_reference, 0.0))
ROI_PARAMETERS.update({"subspin_counts": list(SUBSPIN_COUNTS),
    "subspin_version": SUBSPIN_VERSION,
    "parents_removed_for_subspin_coverage": subspin_report["excluded_parent_count"],
    "selected_pixels_after_subspin_coverage": int(support.sum())})
show_table(list(subspin_report.items()), ("Préparation des sous-spins", "Valeur"))
show_table([("Épaisseur GRE [mm]", reference_metadata["slice_thickness_mm"]),
            ("Épaisseur EPI utilisée [mm]", geometry["slice_thickness_mm"])])

fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
image_panel(axes[0], support_before_subspins, "Parents avant couverture des sous-spins", vmin=0, vmax=1)
image_panel(axes[1], support, "Parents avec tous leurs sous-spins valides", vmin=0, vmax=1)
plot = image_panel(axes[2], subspin_data["b0_sample_spread_per_parent_Hz"],
                   "Étendue du B0 entre sous-spins [Hz]", cmap="magma", vmin=0)
fig.colorbar(plot, ax=axes[2])
show_figure(fig)


In [ ]:
# Le stockage [parent, sous-spin] n'est pas la matrice reconstruite.
model = SubspinACDCModel(
    subspin_data["expanded_experiment"], subspin_data["expanded_calibration"],
    max_current_A=MAX_CURRENT_A, profile=GPU_PROFILE, precision=GPU_PRECISION,
    memory_fraction=GPU_MEMORY_FRACTION, sample_chunk=SAMPLE_CHUNK,
    hardware=HARDWARE, reconstruction_chunk=RECONSTRUCTION_CHUNK,
)
show_table(list(model.runtime_info.items()), ("Modèle effectif", "Valeur"))
print("Courants optimisés :", (model.nc, model.nstates),
      "; sous-spins simulés :", model.runtime_info["subspins"]["active_children"])


### 9b. Vérifier la discrétisation et inspecter le B0 natif

La V2 doit retrouver le signal du 04 à courant nul sur **le même support** :
le B0 et les positions dans le plan n'ont pas changé, et les poids somment à 1.
La V3 possède une cible idéale calculée avec sa propre géométrie de sous-spins.
Le B0 moyen du parent n'est pas utilisé à la place des valeurs individuelles.

In [ ]:
subspin_checks = {"weights_sum_to_one": bool(np.allclose(
    subspin_data["weights"].sum(axis=1), 1, rtol=0, atol=1e-12)),
    "finite_child_B0": bool(np.isfinite(subspin_data["subspin_b0"]).all()),
    "finite_child_ACDC": bool(np.isfinite(subspin_data["fieldbasis"]).all())}
if SUBSPIN_VERSION == "V2" and SUBSPIN_B0_Z_GRADIENT_HZ_PER_M == 0.0:
    # Comparaison sur les parents effectivement conservés, avec la même amplitude.
    point_model = RealisticACDCModel(
        experiment, calibration, max_current_A=MAX_CURRENT_A,
        profile=GPU_PROFILE, precision=GPU_PRECISION,
        memory_fraction=min(GPU_MEMORY_FRACTION, 0.15), sample_chunk=SAMPLE_CHUNK,
        hardware=HARDWARE, reconstruction_chunk=RECONSTRUCTION_CHUNK)
    zero = np.zeros((model.nc, model.nstates))
    _, point_signal = point_model.forward(zero)
    _, child_signal = model.forward(zero)
    zero_signal_error = float(np.linalg.norm(point_signal - child_signal)
                              / max(np.linalg.norm(point_signal), 1e-12))
    subspin_checks["zero_current_matches_04_on_common_support"] = zero_signal_error < 1e-4
    print("Erreur relative V2 / 04 à courant nul :", zero_signal_error)
    del point_model
    if model.device.type == "cuda":
        torch.cuda.empty_cache()
assert all(subspin_checks.values()), subspin_checks
show_table(list(subspin_checks.items()))
print("B0 natif disponible :", b0_result["b0Hz"].shape,
      "; source réellement utilisée :", SUBSPIN_B0_MODE)
print("Parents avec variation B0 entre sous-spins :", subspin_report["parents_with_B0_spread"])

# Exemple : le parent dont l'étendue B0 est la plus grande.
selected_parent = int(np.argmax(np.ptp(subspin_data["subspin_b0"], axis=1)))
selected_flat_index = int(subspin_data["parent_flat_index"][selected_parent])
selected_row, selected_column = np.unravel_index(selected_flat_index, support.shape)
child_b0 = subspin_data["subspin_b0"][selected_parent]
child_x = subspin_data["expanded_experiment"]["geometry"]["X_m"][selected_parent]
child_y = subspin_data["expanded_experiment"]["geometry"]["Y_m"][selected_parent]
child_z_mm = subspin_data["expanded_experiment"]["geometry"]["Z_m"][selected_parent] * 1000
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
image_panel(axes[0], b0_result["b0Hz"], "Carte B0 native signée [Hz]", cmap="RdBu_r")
axes[1].scatter(child_x * 1000, child_y * 1000, c=child_b0, cmap="RdBu_r", s=80)
axes[1].set(xlabel="X [mm]", ylabel="Y [mm]", title=f"Positions du parent [{selected_row}, {selected_column}]")
axes[2].scatter(np.arange(len(child_b0)), child_b0, c=child_z_mm, cmap="viridis")
axes[2].set(xlabel="Indice du sous-spin", ylabel="B0 interpolé [Hz]", title="Valeurs propres à chaque sous-spin")
show_figure(fig)


### 9c. Contraindre les courants réels entre les commandes

Après les limites canal/somme/champ sur les commandes, une contraction globale
impose les limites de la réponse réelle. Les événements comprennent le démarrage,
les intervalles à zéro et l'arrêt final. Avec un τ commun, les maxima de courant
total et de champ sont aux extrémités de chaque segment. Avec des τ différents,
des bornes conservatrices par canal couvrent tout le segment.

Les intégrales de |I| et I² sont analytiques, y compris un passage par zéro.
Les moyennes portent sur le TR complet du `.seq`, pas seulement les ADC.
Le slew physique vaut `(commande - courant réel) / tau` ; le contrôle facultatif
de tension utilise `R*I + L*dI/dt`. Les contractions sont admissibles sans être
une projection orthogonale sur l'ensemble complet des contraintes.

La limite de champ concerne le **champ ACDC généré dans la ROI valide**, pas
le résiduel `B0 + ACDC` ni tout le volume de la tête. Les courants avant la
première commande sont supposés nuls ; le résidu en fin de TR est affiché,
sans supposer un régime périodique établi.

In [ ]:
show_table(list(model.hardware.timing_report.items()))
print("Limites facultatives non renseignées :",
      [name for name in ("max_slew_A_per_s", "avg_abs_current_A", "max_voltage_V", "max_coil_power_W")
       if HARDWARE[name] is None])


### 9d. Initialisation statique puis contraintes dynamiques

Le shim statique fournit un point de départ. On applique ensuite les mêmes
contraintes de forme d'onde que pendant l'optimisation. Cette initialisation
n'est pas exportée comme une expérience indépendante.

In [ ]:
assert IMAGE_INITIALIZATION in ("static", "zero")
if IMAGE_INITIALIZATION == "static":
    static_fit = solve_field_gpu(
        coil_maps[support], -b0_on_reference[support], settings=FIELD_SETTINGS,
        max_current_A=MAX_CURRENT_A, total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
        field_limit_Hz=FIELD_LIMIT_HZ, max_iterations=1000, tolerance=1e-7)
    initial_requested_A = np.repeat(static_fit["currents_A"][:, :1], len(starts), axis=1)
else:
    initial_requested_A = np.zeros((model.nc, model.nstates))
image_initial_currents_A = model.project_currents(
    initial_requested_A, TOTAL_CURRENT_LIMIT_A, FIELD_LIMIT_HZ)
initial_hardware_report = model.hardware.diagnostics(
    image_initial_currents_A, TOTAL_CURRENT_LIMIT_A, FIELD_LIMIT_HZ)
assert initial_hardware_report["all_limits_met"]
show_table(list(initial_hardware_report["checks"].items()), ("Initialisation admissible", "Résultat"))


## 10. Jumeaux numériques MRzero et temps de calcul

La référence idéale et le jumeau avec B0 mesuré utilisent l'aimantation produite
par MRzero. Les ADC sont remis dans l'ordre cartésien de la reconstruction,
indépendamment de l'ordre chronologique des commandes.

La magnitude GRE mesurée sert à définir l'objet. L'objectif reste la MSE relative
de magnitude par rapport au **jumeau idéal MRzero**, sur toute l'image.
Le scan RSS mesuré est affiché séparément avec son échelle propre.

In [ ]:
# Native MRzero magnetization was prepared in section 9.
import time
zero_currents_A = np.zeros_like(image_initial_currents_A)
ideal_image = model.ideal_image
ideal_kdata = model.ideal_kdata
baseline_image, baseline_kdata = model.forward(zero_currents_A)
forward = None
if RUN_FORWARD_MODEL:
    forward = {"ideal_image": ideal_image, "ideal_kdata": ideal_kdata,
               "measured_image": baseline_image, "measured_kdata": baseline_kdata}
    twin_scale = max(np.abs(ideal_image).max(), np.abs(baseline_image).max())
    fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
    image_panel(axes[0], np.abs(ideal_image), "MRzero: ideal zero-B0 twin", vmin=0, vmax=twin_scale)
    image_panel(axes[1], np.abs(baseline_image), "MRzero: measured-B0 twin", vmin=0, vmax=twin_scale)
    image_panel(axes[2], np.abs(np.abs(baseline_image) - np.abs(ideal_image)), "Différence de magnitude", cmap="magma")
    image_panel(axes[3], scan_image, "Scan RSS mesuré : échelle propre")
    show_figure(fig)

baseline_mse = float(np.sum((np.abs(baseline_image) - np.abs(ideal_image))**2)
                     / np.sum(np.abs(ideal_image)**2))
print("Erreur relative d'image sans ACDC :", baseline_mse)

if RUN_BENCHMARK:
    model.objective_and_grad(image_initial_currents_A)  # Échauffement.
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    benchmark_start = time.perf_counter()
    benchmark_mse, _ = model.objective_and_grad(image_initial_currents_A)
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    gradient_seconds = time.perf_counter() - benchmark_start
    show_table([("Objectif + gradient [ms]", f"{gradient_seconds * 1000:.3f}"),
                ("MSE au point initial", benchmark_mse),
                ("Itérations maximales", MAX_ITERATIONS)])

## 11. Optimisation MRzero sur GPU avec suivi visuel réglable

Adam met à jour une commande DAC par canal et par ligne. Les limites par canal,
la somme des valeurs absolues des courants et le champ ACDC sont imposés après
chaque mise à jour. Le modèle garde tout l'historique de la réponse temporelle.

**Pendant l'exécution de la cellule suivante**, la même figure est remplacée
toutes les `LIVE_UPDATE_EVERY` itérations (20 par défaut), ainsi qu'au début
et à la dernière itération. La courbe conserve les valeurs de **toutes** les
itérations, même entre deux affichages. Les échelles des images
restent fixes. La MSE affichée est celle de l'image courante, pas celle d'un
candidat différent. Une MSE relative de 0,01 correspond à une NRMSE de 10 %.

`USE_CUDA_GRAPH = False` : la projection directe applique toutes les contraintes du 04.

L'affichage utilise Matplotlib et `display_id`, sans dépendance à `ipympl`.
La lecture des données et la préparation des figures utilisent le CPU ; le calcul
d'image, du gradient et des projections utilise CUDA. Le temps total inclut
la génération des figures. Le meilleur candidat admissible
est conservé pour les figures finales et les vérifications de la section 12.

In [ ]:
# Réglages du solveur : tous sont regroupés dans la cellule de paramètres.
show_table([("Pas Adam [A]", IMAGE_LEARNING_RATE_A),
            ("Arrêt sur plateau [itérations]", IMAGE_PATIENCE),
            ("Cible MSE facultative", IMAGE_TARGET_MSE)])


In [ ]:
# Refresh the display and solver when rerunning this cell in an existing kernel.
from importlib import reload
import live_optimization
import experiment_optimization
LiveOptimizationPlot = reload(live_optimization).LiveOptimizationPlot
optimize_mrzero_image = reload(experiment_optimization).optimize_mrzero_image

solution = None
if RUN_OPTIMIZATION:
    print("MRI optimization device:", model.device, "| GPU:", model.runtime_info["gpu_name"])
    live_plot = LiveOptimizationPlot(
        reference=ideal_image,
        baseline=baseline_image,
        current_limit_a=MAX_CURRENT_A,
        title="MRzero image optimization: current evaluated candidate",
    )

    def report_progress(snapshot):
        """Display the image, currents and loss from the same evaluated candidate."""
        live_plot.set_history(snapshot["history"]["iteration"], snapshot["history"]["fval"])
        live_plot.update(
            iteration=snapshot["iteration"],
            image=snapshot["image"],
            currents_a=snapshot["currents_A"],
            loss=snapshot["fval"],
            nrmse_percent=100 * np.sqrt(snapshot["fval"]),
            max_field_hz=snapshot["max_field_Hz"],
            field_limit_hz=FIELD_LIMIT_HZ,
            phase="Projected Adam",
        )

    try:
        solution = optimize_mrzero_image(
            model, image_initial_currents_A,
            total_current_limit_A=TOTAL_CURRENT_LIMIT_A,
            field_limit_Hz=FIELD_LIMIT_HZ,
            max_iterations=MAX_ITERATIONS,
            learning_rate_A=IMAGE_LEARNING_RATE_A,
            target_mse=IMAGE_TARGET_MSE,
            patience=IMAGE_PATIENCE,
            callback=report_progress,
            callback_every=LIVE_UPDATE_EVERY,
            use_cuda_graph=USE_CUDA_GRAPH,
        )
    except BaseException:
        live_plot.finish("Stopped: last evaluated candidate remains visible")
        raise
    else:
        info = solution["optimization_info"]
        live_plot.finish(f"{info['status']}; best feasible MSE = {info['fval']:.6g}")

    show_table(list(info.items()), ("Solver result", "Value"))
    show_table([
        ("Initialization", IMAGE_INITIALIZATION),
        ("Initial relative MSE", solution["initial_relative_mse"]),
        ("Best feasible relative MSE", info["fval"]),
        ("Best feasible NRMSE [%]", 100 * np.sqrt(info["fval"])),
        ("Live display and snapshot time [s]", info["callback_seconds"]),
        ("Optimization excluding live display [s]", info["seconds_without_callback"]),
    ])
    history = solution["history"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    for ax, key, title in zip(axes, ("fval", "firstorderopt", "constrviolation"),
                             ("Relative image MSE", "Projected step [A]", "Constraint violation")):
        ax.plot(history["iteration"], history[key], ".-")
        ax.set(xlabel="Iteration", ylabel=title, title=title)
    axes[0].plot(history["iteration"], history["best_fval"], "--", label="Best feasible MSE")
    axes[0].legend()
    show_figure(fig)
else:
    print("Image optimization disabled in the parameter cell.")

## 12. Résultat et contrôles après conversion nominale DAC

La conversion du lecteur de formes d'onde existant est conservée : conversion
signée 16 bits sur ±5 A, par troncature. La résolution effective de 0.6 mA du
manuscrit n'est pas substituée à cette règle. On recalcule les contraintes sur
les commandes et la réponse physique après conversion, ainsi que la MSE.
Le CSV reste en ampères.

In [ ]:
solver_ok = False
if solution is not None:
    currents = solution["currents_A"]
    quantized_currents_A = quantize_currents(currents)
    hardware_report = model.hardware.diagnostics(currents, TOTAL_CURRENT_LIMIT_A, FIELD_LIMIT_HZ)
    quantized_hardware_report = model.hardware.diagnostics(
        quantized_currents_A, TOTAL_CURRENT_LIMIT_A, FIELD_LIMIT_HZ)
    quantized_image, _ = model.forward(quantized_currents_A)
    quantized_mse = float(np.sum((abs(quantized_image) - abs(ideal_image))**2)
                          / np.sum(abs(ideal_image)**2))
    solver_ok = bool(hardware_report["all_limits_met"]
                     and quantized_hardware_report["all_limits_met"]
                     and np.isfinite(info["fval"])
                     and np.isfinite(quantized_mse)
                     and quantized_mse <= solution["initial_relative_mse"] + 1e-7
                     and info["fval"] <= solution["initial_relative_mse"] + 1e-7)
    show_table([(name, passed, quantized_hardware_report["checks"][name])
                for name, passed in hardware_report["checks"].items()],
               ("Contrôle", "Optimisé", "Après conversion DAC"))
    show_table([
        ("MSE initiale", solution["initial_relative_mse"]),
        ("MSE optimisée", info["fval"]),
        ("MSE après conversion nominale DAC", quantized_mse),
        ("Maximum somme |I réel|, borne [A]", hardware_report["peak_total_current_bound_A"]),
        ("Maximum champ ACDC réel, borne [Hz]", hardware_report["peak_field_bound_Hz"]),
        ("Maximum slew physique [A/s]", max(hardware_report["peak_slew_A_per_s"])),
        ("Résidu maximal en fin de TR [A]", max(abs(np.asarray(hardware_report["end_current_A"])))),
        ("Contrôles numériques d'export satisfaits", solver_ok),
    ])
    fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
    images = (ideal_image, baseline_image, solution["image"], quantized_image)
    titles = ("Cible B0 nul", "Sans ACDC", "ACDC optimisé", "Après conversion DAC")
    vmax = max(abs(image).max() for image in images)
    for ax, image, title in zip(axes, images, titles):
        image_panel(ax, abs(image), title, vmin=0, vmax=vmax)
    show_figure(fig)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
    plot = axes[0].imshow(currents, cmap="RdBu_r", aspect="auto",
                          vmin=-MAX_CURRENT_A, vmax=MAX_CURRENT_A)
    axes[0].set(xlabel="État chronologique", ylabel="Canal", title="Commandes DAC [A]")
    fig.colorbar(plot, ax=axes[0])
    axes[1].bar(np.arange(1, model.nc + 1), hardware_report["avg_abs_current_A"])
    axes[1].set(xlabel="Canal", ylabel="Moyenne |I réel| [A]", title="Moyenne sur le TR complet")
    if "average_coil_power_W" in hardware_report:
        heating = hardware_report["average_coil_power_W"]
        heating_label = "Puissance résistive moyenne [W]"
    else:
        heating = hardware_report["mean_square_current_A2"]
        heating_label = "Moyenne I² [A²] - résistance non renseignée"
    axes[2].bar(np.arange(1, model.nc + 1), heating)
    axes[2].set(xlabel="Canal", ylabel=heating_label, title="Diagnostic de dissipation")
    show_figure(fig)


### 12b. Phases individuelles avant la somme du signal

Les figures suivantes montrent les phases **B0 + ACDC de chaque sous-spin**
d'un parent, sans les gradients d'encodage. La cohérence est le module de la
somme complexe pondérée : 1 signifie des phases identiques, 0 une annulation.
Ces courbes sont un diagnostic local ; l'objectif utilise le signal complet
à tous les ADC et la reconstruction continue.

In [ ]:
if solution is not None:
    response = solution["response_diagnostics"]
    adc_time_s = np.asarray(response["adc_time_s"]).ravel()
    chronological = np.argsort(adc_time_s)
    actual_adc = np.asarray(response["actual_currents_at_adc_A"])
    commanded_adc = np.asarray(response["commanded_currents_at_adc_A"])
    integrals = np.asarray(response["integrated_currents_to_adc_As"])
    parent = selected_parent
    local_b0 = subspin_data["subspin_b0"][parent]
    local_basis = subspin_data["fieldbasis"][parent]
    local_weights = subspin_data["weights"][parent]
    cycles_without = local_b0[:, None] * adc_time_s[None, :]
    cycles_with = cycles_without + local_basis @ integrals
    relative_cycles = cycles_with - np.sum(cycles_with * local_weights[:, None], axis=0)
    coherence_without = abs(np.sum(local_weights[:, None] * np.exp(-2j * np.pi * cycles_without), axis=0))
    coherence_with = abs(np.sum(local_weights[:, None] * np.exp(-2j * np.pi * cycles_with), axis=0))
    fig, axes = plt.subplots(1, 3, figsize=(17, 4), layout="constrained")
    plot = axes[0].imshow(actual_adc[:, chronological], aspect="auto", origin="lower",
        cmap="RdBu_r", vmin=-MAX_CURRENT_A, vmax=MAX_CURRENT_A,
        extent=[adc_time_s.min()*1000, adc_time_s.max()*1000, 0.5, model.nc+0.5])
    axes[0].set(xlabel="Temps depuis RF [ms]", ylabel="Canal", title="Courants réels aux ADC")
    fig.colorbar(plot, ax=axes[0], label="Courant [A]")
    axes[1].plot(adc_time_s[chronological]*1000,
                 (-2*np.pi*relative_cycles[:, chronological]).T, alpha=0.5)
    axes[1].set(xlabel="Temps depuis RF [ms]", ylabel="Phase relative [rad]",
                title="Phase B0 + ACDC de chaque sous-spin")
    axes[2].plot(adc_time_s[chronological]*1000, coherence_without[chronological], label="Sans ACDC")
    axes[2].plot(adc_time_s[chronological]*1000, coherence_with[chronological], label="ACDC optimisé")
    axes[2].set(xlabel="Temps depuis RF [ms]", ylabel="Cohérence [0, 1]",
                ylim=(-0.02,1.02), title=f"Parent [{selected_row}, {selected_column}]")
    axes[2].legend()
    show_figure(fig)

    instantaneous_image = response["instantaneous_image"]
    comparison_scale = max(abs(instantaneous_image).max(), abs(solution["image"]).max())
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    image_panel(axes[0], abs(instantaneous_image), "Même commande : réponse instantanée", vmin=0, vmax=comparison_scale)
    image_panel(axes[1], abs(solution["image"]), "Même commande : réponse filtrée", vmin=0, vmax=comparison_scale)
    plot = image_panel(axes[2], abs(solution["image"])-abs(instantaneous_image),
                       "Différence due au filtre", cmap="RdBu_r")
    fig.colorbar(plot, ax=axes[2])
    show_figure(fig)


## 12c. Preview the predicted real waveform

This figure shows the **complete current waveform predicted by the selected temporal
response**, including every imaging-line transition and the decay after the final zero
command. MRzero/PyTorch uses the optimized DAC setpoints, the sequence command windows and
the saved channel time constants. The initial real current is assumed to be zero.
This preview can be run on an existing solution without repeating optimization.

The first panel displays all physical outputs. The second panel zooms into a transition
so the first-order rise or fall is visible. `WAVEFORM_PREVIEW_CHANNEL` uses the 1-based
output numbers in `channel_map`; `None` selects the largest command change.
The channel order, polarity and time origin follow the settings used in section 13.
With the temporal model disabled, this preview follows instantaneous commands.

The waveform is a model prediction requiring experimental validation. Section 13
below keeps its existing DAC-command preview and CSV export.

In [ ]:
# Sample the full predicted waveform in MRzero/PyTorch; do not optimize or export here.
WAVEFORM_PREVIEW_CHANNEL = None  # Configured output number (1-based), or None for the largest jump.

if solution is None:
    print("A MRzero/PyTorch solution is needed for the predicted waveform preview.")
else:
    from export_acdc_waveform import validate_waveform_settings

    assert np.isscalar(RF_DELAY_S) and np.isfinite(RF_DELAY_S) and RF_DELAY_S >= 0, "RF_DELAY_S must be finite and nonnegative."
    preview_currents = np.asarray(solution["currents_A"])
    preview_settings = validate_waveform_settings(WAVEFORM_PARAMETERS, preview_currents.shape[0])
    waveform_preview = model.sample_waveform(preview_currents)
    model_time_s = np.asarray(waveform_preview["time_s"]).ravel()
    predicted_current = np.asarray(waveform_preview["actual_A"])
    commanded_current = np.asarray(waveform_preview["commanded_A"])

    # Apply the same output numbering and polarity as the CSV; unused outputs stay zero.
    output_numbers = np.asarray(preview_settings["channel_map"])
    polarity = np.asarray(preview_settings["channel_polarity"])[:, None]
    output_count = ((int(output_numbers.max()) + 7) // 8) * 8
    physical_real = np.zeros((output_count, model_time_s.size))
    physical_command = np.zeros_like(physical_real)
    physical_real[output_numbers - 1] = predicted_current * polarity
    physical_command[output_numbers - 1] = commanded_current * polarity
    plot_time_s = model_time_s + RF_DELAY_S
    if RF_DELAY_S > 0:
        plot_time_s = np.r_[0.0, plot_time_s]
        physical_real = np.column_stack((np.zeros(output_count), physical_real))
        physical_command = np.column_stack((np.zeros(output_count), physical_command))

    # Include the first command and final return to zero when choosing the zoom.
    jumps = np.abs(np.diff(np.column_stack((np.zeros(preview_currents.shape[0]), commanded_current)), axis=1))
    if WAVEFORM_PREVIEW_CHANNEL is None:
        channel, transition_sample = np.unravel_index(np.argmax(jumps), jumps.shape)
    else:
        assert WAVEFORM_PREVIEW_CHANNEL in output_numbers, "Choose an output number present in channel_map."
        channel = int(np.flatnonzero(output_numbers == WAVEFORM_PREVIEW_CHANNEL)[0])
        transition_sample = int(np.argmax(jumps[channel]))
    output_index = int(output_numbers[channel] - 1)
    transition_s = model_time_s[transition_sample] + RF_DELAY_S
    saved_tau_s = np.asarray(solution["current_response"]["tau_s"]).ravel()
    channel_tau_s = float(saved_tau_s[0] if saved_tau_s.size == 1 else saved_tau_s[channel])
    zoom = np.abs(plot_time_s - transition_s) <= 5 * channel_tau_s + 1e-12
    origin = "first RF center" if RF_DELAY_S == 0 else "external trigger"
    response_enabled = bool(solution["current_response"]["enabled"])

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), layout="constrained")
    axes[0].plot(plot_time_s * 1000, physical_real.T, alpha=0.65, linewidth=0.8)
    axes[0].axvline((ends[-1] + RF_DELAY_S) * 1000, color="black", linestyle="--", label="Final zero command")
    axes[0].set(xlabel=f"Time from {origin} [ms]", ylabel="Physical output current [A]",
                title="Complete predicted current waveform" if response_enabled else "Complete waveform: temporal response disabled")
    axes[0].legend()
    zoom_time_us = (plot_time_s[zoom] - transition_s) * 1e6
    axes[1].step(zoom_time_us, physical_command[output_index, zoom], where="post",
                 color="gray", linestyle="--", label="DAC command")
    axes[1].plot(zoom_time_us, physical_real[output_index, zoom], label="Predicted real current")
    axes[1].set(xlabel="Time from selected command change [us]", ylabel="Current [A]",
                title=f"Output {output_index + 1}: transition at {transition_s * 1000:.3f} ms; tau = {channel_tau_s * 1e6:.2f} us")
    axes[1].legend()
    show_figure(fig)

### 12d. Cohérence intravoxel au TE : signal de champ seul

Cette carte isole la dispersion de phase B0 + ACDC à TE, en excluant l'encodage
spatial et la relaxation. Elle conserve le volume et les poids de chaque parent.
Elle n'est ni l'image reconstruite ni un second objectif d'optimisation.
La phase ACDC utilise l'intégrale complète des courants réels depuis la RF.

In [ ]:
coherence_report = {}
if solution is not None:
    from acdc_gpu import response_matrices
    h_te, _ = response_matrices(starts, ends, [acquisition["TE_s"]], CURRENT_RESPONSE["tau_s"])
    integral_te = np.einsum("cs,cts->ct", currents, h_te)[:, 0]
    child_b0_all = subspin_data["subspin_b0"]
    cycles_te_without = child_b0_all * acquisition["TE_s"]
    cycles_te_with = cycles_te_without + np.einsum("pqc,c->pq", subspin_data["fieldbasis"], integral_te)
    weights = subspin_data["weights"]
    def field_coherence(cycles):
        # Retirer une phase commune ne change pas le module et stabilise l'affichage.
        centered = cycles - np.sum(cycles * weights, axis=1, keepdims=True)
        return abs(np.sum(weights * np.exp(-2j * np.pi * centered), axis=1))
    coherence_te_without = field_coherence(cycles_te_without)
    coherence_te_with = field_coherence(cycles_te_with)
    def parent_map(values):
        image = np.full(support.shape, np.nan)
        image.ravel()[subspin_data["parent_flat_index"]] = values
        return image
    coherence_te_without_image = parent_map(coherence_te_without)
    coherence_te_with_image = parent_map(coherence_te_with)
    coherence_report = {"TE_s": float(acquisition["TE_s"]),
        "baseline_min": float(coherence_te_without.min()),
        "baseline_median": float(np.median(coherence_te_without)),
        "optimized_min": float(coherence_te_with.min()),
        "optimized_median": float(np.median(coherence_te_with)),
        "diagnostic": "field-only weighted complex coherence; excludes spatial encoding and relaxation"}
    show_table(list(coherence_report.items()))
    fig, axes = plt.subplots(1,3,figsize=(15,4),layout="constrained")
    for ax, values, title in zip(axes[:2], (coherence_te_without_image,coherence_te_with_image),
                                  ("Cohérence sans ACDC au TE", "Cohérence avec ACDC au TE")):
        plot=image_panel(ax,values,title,cmap="viridis",vmin=0,vmax=1)
        fig.colorbar(plot,ax=ax)
    difference=coherence_te_with_image-coherence_te_without_image
    scale=max(float(np.nanmax(abs(difference))),1e-6)
    plot=image_panel(axes[2],difference,"Différence de cohérence au TE",cmap="RdBu_r",vmin=-scale,vmax=scale)
    fig.colorbar(plot,ax=axes[2])
    show_figure(fig)


### 12e. Isoler l'effet du choix de reconstruction

Le **même signal MRzero** est reconstruit par l'IFFT du 03 et par R. Chaque
image est comparée à la référence reconstruite par son propre opérateur.
Ce n'est pas une réoptimisation du 03. Sur une grille presque cartésienne,
un écart nul ou faible est attendu ; on ne promet pas de gain artificiel.

In [ ]:
if solution is not None:
    def old_ifft(values):
        return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(values)))
    def relative_image_mse(image, reference):
        return float(np.sum((abs(image) - abs(reference))**2) / np.sum(abs(reference)**2))
    old_reference = old_ifft(ideal_kdata)
    old_baseline = old_ifft(baseline_kdata)
    old_corrected = old_ifft(solution["kdata"])
    reconstruction_comparison = {
        "same_signal_baseline_old_ifft_mse": relative_image_mse(old_baseline, old_reference),
        "same_signal_baseline_continuous_mse": baseline_mse,
        "same_signal_optimized_old_ifft_mse": relative_image_mse(old_corrected, old_reference),
        "same_signal_optimized_continuous_mse": info["fval"],
        "complex_relative_image_difference": float(np.linalg.norm(old_corrected - solution["image"])
                                                   / np.linalg.norm(solution["image"])),
    }
    show_table(list(reconstruction_comparison.items()))
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    vmax = max(abs(old_corrected).max(), abs(solution["image"]).max())
    image_panel(axes[0], abs(old_corrected), "Même signal : IFFT du 03", vmin=0, vmax=vmax)
    image_panel(axes[1], abs(solution["image"]), "Même signal : reconstruction R", vmin=0, vmax=vmax)
    plot = image_panel(axes[2], abs(solution["image"]) - abs(old_corrected),
                       "Différence de magnitude R - IFFT", cmap="RdBu_r")
    fig.colorbar(plot, ax=axes[2])
    show_figure(fig)


## 13. Sauvegarde et export CSV facultatif

Courants, images, historiques, paramètres et sources sont sauvegardés dans
`results_experiment_realistic_v3/`. Le CSV contient les **commandes**, pas les
courants déjà filtrés. Les temps restent espacés d'au moins 8 µs.
Aucune interface matérielle n'est pilotée.

In [ ]:
if solution is not None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        OUTPUT_DIR / "mrzero04_v3_optimization.npz",
        reference=ideal_image, baseline=baseline_image, optimized=solution["image"],
        currents_A=currents, quantized_currents_A=quantized_currents_A,
        initial_currents_A=solution["initial_currents_A"],
        iteration=history["iteration"], relative_mse=history["fval"],
        best_relative_mse=history["best_fval"],
        kx_mInv=acquisition["kx_mInv"], ky_mInv=acquisition["ky_mInv"],
        adc_time_s=acquisition["time_s"], X_m=X_m, Y_m=Y_m, simulation_mask=support,
        subspin_parent_flat_index=subspin_data["parent_flat_index"],
        subspin_weights=subspin_data["weights"],
        subspin_xyz_patient_m=subspin_data["xyz_m"],
        subspin_offsets_patient_m=subspin_data["offsets_m"],
        subspin_X_m=subspin_data["expanded_experiment"]["geometry"]["X_m"],
        subspin_Y_m=subspin_data["expanded_experiment"]["geometry"]["Y_m"],
        subspin_Z_m=subspin_data["expanded_experiment"]["geometry"]["Z_m"],
        subspin_B0_Hz=subspin_data["subspin_b0"],
        subspin_basis_Hz_per_A=subspin_data["fieldbasis"],
        subspin_amplitude=subspin_data["expanded_experiment"]["m0_proxy"],
        native_B0_Hz=b0_result["b0Hz"],native_B0_X_m=b0_result["X_m"],native_B0_Y_m=b0_result["Y_m"],
        native_B0_mask=b0_result["mask"],
        coherence_TE_without_ACDC=coherence_te_without_image,
        coherence_TE_optimized=coherence_te_with_image)
    result_summary = {
        "optimization": info, "runtime": model.runtime_info,
        "subspins": subspin_report, "subspin_checks": subspin_checks,
        "coherence_TE": coherence_report,
        "current_preset": CURRENT_PRESET, "max_current_A": MAX_CURRENT_A,
        "total_current_limit_A": TOTAL_CURRENT_LIMIT_A, "hardware_configuration": HARDWARE,
        "current_response_tau_s": CURRENT_RESPONSE["tau_s"],
        "hardware": hardware_report, "hardware_after_DAC_conversion": quantized_hardware_report,
        "quantized_relative_mse": quantized_mse,
        "reconstruction": model.reconstruction.report,
        "reconstruction_comparison": reconstruction_comparison,
        "measured_reconstruction": {"GRE": reference_reconstruction_report,
                                    "EPI": measured_scan_reconstruction_report,
                                    "B0_echoes": b0_reconstruction_reports},
        "source_sha256": source_hashes, "paper_sha256": paper_hashes,
        "sources": {"controller": "main_revised_small.pdf, PDF pp. 1, 2, 6, 7, 9",
                    "chapter6": "UHF_Neuro_MRI_book_CH6 (2).pdf, PDF p. 11 / printed p. 85"},
        "roi_parameters": ROI_PARAMETERS, "export_checks_passed": solver_ok,
    }
    (OUTPUT_DIR / "mrzero04_v3_optimization.json").write_text(
        json.dumps(result_summary, default=json_value, indent=2), encoding="utf-8")
    print("Résultats :", OUTPUT_DIR)


In [ ]:
# Option de la première colonne du CSV.
CSV_ZERO_TIME_COLUMN = False  # True : des 0 ; False : temps en secondes

# Recharger l'export pour utiliser la numérotation dans un noyau déjà ouvert.
import importlib
import export_acdc_waveform
export_waveform = importlib.reload(export_acdc_waveform).export_waveform

# Generate data only. The FPGA is operated separately from this notebook.
csv_file = None
if EXPORT_WAVEFORM and solver_ok:
    assert np.isscalar(RF_DELAY_S) and np.isfinite(RF_DELAY_S) and RF_DELAY_S >= 0, "RF_DELAY_S must be finite and nonnegative."
    if timing["sequence_kind"] == "gre":
        csv_edges = np.column_stack((starts, ends)).ravel() + RF_DELAY_S
        csv_currents = np.zeros((currents.shape[0], 2 * currents.shape[1] - 1))
        csv_currents[:, ::2] = currents
        for start, end in zip(starts, ends):
            overlaps_rf = (np.asarray(timing["rf_start_s"]) < end - 1e-12) & (np.asarray(timing["rf_end_s"]) > start + 1e-12)
            assert not overlaps_rf.any(), "A current window overlaps a GRE RF pulse."
    else:
        csv_edges = np.r_[starts, ends[-1]] + RF_DELAY_S
        csv_currents = currents
    try:
        csv_export = export_waveform(
            csv_currents, csv_edges, WAVEFORM_PARAMETERS, OUTPUT_DIR,
            zero_time_column=CSV_ZERO_TIME_COLUMN,
            numbered_filename=True,
        )
        csv_file = Path(csv_export["csv_file"])
    except (ValueError, TypeError, FileExistsError) as error:
        print("CSV export blocked:", error)
else:
    print("CSV export disabled or numerical solution not eligible.")

if csv_file is not None:
    waveform = np.loadtxt(csv_file, delimiter=",")
    # Le graphique conserve les vrais temps dans les deux modes d'export.
    preview_times = np.r_[0.0, csv_edges] if csv_edges[0] > 0 else csv_edges
    fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
    ax.step(preview_times * 1000, waveform[:, 1:], where="post", alpha=0.6)
    origin = "first RF center" if RF_DELAY_S == 0 else "external trigger"
    ax.set(xlabel=f"Time from {origin} [ms]", ylabel="Physical output current [A]", title="Exported waveform: initial and final zero currents")
    show_figure(fig)
    print("New exported file:", csv_file)
    print("Première colonne :", "uniquement des 0" if CSV_ZERO_TIME_COLUMN else "temps en secondes")
print("Figures et résultats conservés dans le notebook et la mémoire Python.")

## 14. Comparaison avec un scan corrigé mesuré

Cette comparaison reprend le 03 avec la même reconstruction continue par
récepteur avant RSS. Le fichier corrigé doit avoir été acquis avec la forme
d'onde à évaluer : un ancien fichier ne valide pas les courants du 04.
Le gain d'intensité est ajusté uniquement sur la baseline, puis appliqué sans
changement au scan corrigé.

In [ ]:
# Optional measured comparison: explicitly choose one corrected acquisition.
RESULTS_DIR = INPUT_DIR / "results"
CORRECTED_SCAN_NAME = "meas_MID00351_FID46584_scan.dat"  # Example: "meas_MID00351_FID46584_scan.dat"
CORRECTED_SCAN_FILE = RESULTS_DIR / (CORRECTED_SCAN_NAME or "scan_corr.dat")
CORRECTED_SEQUENCE_FILE = FILES["scan_sequence"]
RMSE_USE_ROI = True
NORMALIZE_COMPARISON = True
print("Available result scans:", [path.name for path in sorted(RESULTS_DIR.glob("*.dat"))])
print("Selected corrected scan:", CORRECTED_SCAN_NAME)

### 14a. Reconstruct the corrected acquisition and verify its sequence

`read_scan` handles the Twix `REFLECT` and `PHASCOR` flags and combines all receivers
with RSS. Its navigator fit comes from `scan_corr.dat`; the baseline fit is not reused.
The same `NAVIGATOR_PHASE_BRANCH` is applied to both EPI scans.
The corrected acquisition must match its own `.seq`. The three scans must describe the
same slice center and orientation before a pixelwise RMSE can be computed.

The before/after panels below diagnose navigator processing within the corrected
acquisition. They are not the comparison of ACDC off versus ACDC on.

In [ ]:
measured_corrected_image = None
if not CORRECTED_SCAN_FILE.is_file():
    print("Comparison skipped: choose CORRECTED_SCAN_NAME in the preceding cell.")
    print("Then rerun section 14 from its parameter cell.")
else:
    assert CORRECTED_SCAN_FILE.suffix.lower() == ".dat", "The corrected scan must be raw Siemens Twix data (.dat)."
    assert CORRECTED_SEQUENCE_FILE.is_file(), f"Missing corrected sequence: {CORRECTED_SEQUENCE_FILE}"
    measured_corrected_acquisition, measured_corrected_timing = read_sequence(CORRECTED_SEQUENCE_FILE)
    measured_corrected_candidate, measured_corrected_metadata, measured_corrected_diagnostics = read_scan(
        CORRECTED_SCAN_FILE, measured_corrected_acquisition, return_intermediates=True,
        navigator_phase_branch=NAVIGATOR_PHASE_BRANCH)
    measured_corrected_candidate, measured_corrected_reconstruction_report = reconstruct_receivers(
        measured_corrected_diagnostics, measured_corrected_acquisition)
    measured_corrected_candidate = rotate_input_scan(measured_corrected_candidate)
    rotate_scan_diagnostics(measured_corrected_diagnostics)
    assert np.isfinite(measured_corrected_candidate).all() and measured_corrected_candidate.max() > 0, "Corrected reconstruction has invalid or zero signal."

    corrected_checks = []
    for key in ("TE_s", "TR_s", "dwell_s"):
        tolerance = 1e-9 if key == "dwell_s" else 1e-6
        corrected_checks.append((key, abs(measured_corrected_metadata[key] - measured_corrected_acquisition[key]) <= tolerance))
    corrected_checks += [
        ("FOV", np.allclose(measured_corrected_metadata["FOV_m"], measured_corrected_acquisition["FOV_m"], rtol=1e-4, atol=1e-6)),
        ("Navigator count", measured_corrected_metadata["navigator_count"] == measured_corrected_timing["navigator_count"]),
        ("Readout polarity", np.array_equal(measured_corrected_metadata["polarity"], measured_corrected_acquisition["polarity"])),
        ("Chronological phase rows", measured_corrected_metadata["acquired_phase_rows"] == np.argsort(measured_corrected_acquisition["line_index"][:, 0]).tolist()),
    ]
    show_table([(name, "PASS" if passed else "FAIL") for name, passed in corrected_checks], ("Corrected acquisition check", "Result"))
    assert all(passed for _, passed in corrected_checks), "Corrected Twix and sequence disagree; inspect the checks above."

    for label, metadata in (("Baseline", scan_metadata), ("Corrected", measured_corrected_metadata)):
        assert np.allclose(metadata["slice_position_mm"], reference_metadata["slice_position_mm"], rtol=0, atol=1e-3), f"{label} and reference slice centers differ; register the acquisitions before RMSE."
        assert np.isclose(abs(np.dot(metadata["slice_quaternion_wxyz"], reference_metadata["slice_quaternion_wxyz"])), 1, rtol=0, atol=1e-8), f"{label} and reference orientations differ; register the acquisitions before RMSE."

    measured_corrected_image = measured_corrected_candidate
    measured_metadata = (reference_metadata, scan_metadata, measured_corrected_metadata)
    show_table([
        ("Matrix [phase, readout]", *[metadata["image_shape"] for metadata in measured_metadata]),
        ("TE [ms]", *[metadata["TE_s"] * 1000 for metadata in measured_metadata]),
        ("TR [ms]", *[metadata["TR_s"] * 1000 for metadata in measured_metadata]),
        ("Slice thickness [mm]", *[metadata["slice_thickness_mm"] for metadata in measured_metadata]),
        ("Navigators", *[metadata["navigator_count"] for metadata in measured_metadata]),
    ], ("Measured parameter", "GRE reference", "Without ACDC", "With ACDC"))

    if measured_corrected_metadata["navigator_count"]:
        before_nav = measured_corrected_diagnostics["uncorrected_image"]
        nav_scale = max(before_nav.max(), measured_corrected_image.max())
        fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
        image_panel(axes[0], before_nav, "ACDC on: before navigator correction", vmin=0, vmax=nav_scale)
        image_panel(axes[1], measured_corrected_image, "ACDC on: after navigator correction", vmin=0, vmax=nav_scale)
        plot = image_panel(axes[2], measured_corrected_diagnostics["navigator_phase_correction_rad"], "ACDC on: fitted navigator phase [rad]", cmap="RdBu_r")
        axes[2].set(xlabel="Readout column", ylabel="Receiver index")
        fig.colorbar(plot, ax=axes[2])
        show_figure(fig)
    else:
        print("Corrected scan has no navigators; no navigator phase correction is applied.")

### 14b. Put both scans on the simulation grid and show the RMSE mask

The native image grids come from their Twix matrix, FOV and slice center. The existing
`resample_to_grid` function samples each scan on the GRE physical grid when needed.
This is linear interpolation of already aligned images, with **no registration or extrapolation**.
Identical grids retain their original values.

Both errors use exactly the same pixels: the chosen ROI intersected with valid coverage
from both scans. The table reports any excluded pixels; this comparison mask does not
change the ROI used by MRzero/PyTorch or the exported waveform.

In [ ]:
def measured_scan_on_reference(image, metadata):
    # Axial readout = patient +y; phase = patient -x, as verified in step 14a.
    ny, nx = image.shape
    dx, dy = np.asarray(metadata["FOV_m"]) / [nx, ny]
    sag, cor, _ = np.asarray(metadata["slice_position_mm"]) / 1000
    x = (np.arange(nx) - nx // 2) * dx + cor
    y = (np.arange(ny) - ny // 2) * dy - sag
    scan_x, scan_y = np.meshgrid(x, y)
    return resample_to_grid(image, scan_x, scan_y, np.isfinite(image), X_m, Y_m)


if measured_corrected_image is not None:
    measured_baseline_on_reference, measured_baseline_valid = measured_scan_on_reference(scan_image, scan_metadata)
    measured_corrected_on_reference, measured_corrected_valid = measured_scan_on_reference(measured_corrected_image, measured_corrected_metadata)
    requested_rmse_mask = support.copy() if RMSE_USE_ROI else np.ones(reference_magnitude.shape, dtype=bool)
    common_image_coverage = measured_baseline_valid & measured_corrected_valid & np.isfinite(reference_magnitude)
    measured_rmse_mask = requested_rmse_mask & common_image_coverage
    assert measured_rmse_mask.any(), "No common valid pixels remain for RMSE; inspect the image coverage and ROI."
    show_table([
        ("Requested region", "Final optimization ROI" if RMSE_USE_ROI else "Full reference image"),
        ("Requested pixels", int(requested_rmse_mask.sum())),
        ("Pixels excluded for missing scan coverage", int((requested_rmse_mask & ~common_image_coverage).sum())),
        ("Common pixels used for both RMSE values", int(measured_rmse_mask.sum())),
    ])
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    image_panel(axes[0], requested_rmse_mask, "Requested RMSE region", vmin=0, vmax=1)
    image_panel(axes[1], common_image_coverage, "Common valid image coverage", vmin=0, vmax=1)
    image_panel(axes[2], reference_magnitude, "GRE: green = pixels used for both errors")
    axes[2].imshow(np.ma.masked_where(~measured_rmse_mask, measured_rmse_mask), cmap="Greens", vmin=0, vmax=1, alpha=0.45, interpolation="nearest")
    show_figure(fig)

### 14c. Measure RMSE and compare the three measured images

With `NORMALIZE_COMPARISON = True`, a single positive least-squares intensity gain is
estimated **only from the scan without ACDC** on the common RMSE mask:
`gain = sum(reference * baseline) / sum(baseline**2)`.
The same gain is applied unchanged to the scan with ACDC. All three images are then
divided by the GRE maximum within that mask. The RMSE and the final figure use these
same images, with one shared intensity scale. The corrected scan is never fitted or
normalized independently.

`RMSE = sqrt(mean((scan - reference)**2))` on the displayed mask.
`NRMSE [%] = 100 * RMSE / sqrt(mean(reference**2))` on that same mask.
Set `NORMALIZE_COMPARISON = False` to keep raw RSS intensities: RMSE is then in arbitrary
Twix intensity units. NRMSE is dimensionless in either mode. Native reconstructions remain unchanged.

These are **measured image differences**, separate from MRzero/PyTorch's simulated optimization
MSE. GRE and EPI can differ in TE, TR, contrast, receive gain and slice thickness; those
differences also contribute to RMSE. A lower value measures closer image similarity
under this comparison, and does not isolate the shimming error.

In [ ]:
if measured_corrected_image is not None:
    reference_roi = reference_magnitude[measured_rmse_mask]
    baseline_roi = measured_baseline_on_reference[measured_rmse_mask]
    assert np.max(reference_roi) > 0 and np.sum(baseline_roi**2) > 0, "Reference and baseline need nonzero signal in the RMSE mask."
    measured_intensity_gain = float(np.dot(reference_roi, baseline_roi) / np.dot(baseline_roi, baseline_roi)) if NORMALIZE_COMPARISON else 1.0
    measured_reference_scale = float(reference_roi.max()) if NORMALIZE_COMPARISON else 1.0
    assert np.isfinite(measured_intensity_gain) and measured_intensity_gain > 0, "The baseline-to-reference intensity gain must be positive and finite."

    measured_comparison_images = [
        reference_magnitude / measured_reference_scale,
        measured_intensity_gain * measured_baseline_on_reference / measured_reference_scale,
        measured_intensity_gain * measured_corrected_on_reference / measured_reference_scale,
    ]
    reference_for_rmse = measured_comparison_images[0][measured_rmse_mask]
    reference_rms = np.sqrt(np.mean(reference_for_rmse**2))
    measured_rmse = [float(np.sqrt(np.mean((image[measured_rmse_mask] - reference_for_rmse)**2))) for image in measured_comparison_images[1:]]
    measured_nrmse_percent = [100 * error / reference_rms for error in measured_rmse]
    rmse_units = "GRE-normalized units" if NORMALIZE_COMPARISON else "Twix RSS units"
    show_table([
        ("Shared baseline-to-reference gain", f"{measured_intensity_gain:.6g}"),
        ("Reference normalization scale", f"{measured_reference_scale:.6g}"),
        ("RMSE units", rmse_units),
    ])
    show_table([
        (label, f"{error:.6f}", f"{relative_error:.3f}")
        for label, error, relative_error in zip(("Without ACDC", "With ACDC"), measured_rmse, measured_nrmse_percent)
    ], ("Measured scan vs GRE", "RMSE", "NRMSE [%]"))

    # Show the same physical grid and intensity scaling used in the error calculation.
    dx_mm = (X_m[0, 1] - X_m[0, 0]) * 1000
    dy_mm = (Y_m[1, 0] - Y_m[0, 0]) * 1000
    comparison_extent = [X_m.min() * 1000 - dx_mm / 2, X_m.max() * 1000 + dx_mm / 2,
                         Y_m.min() * 1000 - dy_mm / 2, Y_m.max() * 1000 + dy_mm / 2]
    comparison_max = max(np.nanmax(image) for image in measured_comparison_images)
    titles = ["GRE reference", f"Scan without ACDC\nRMSE = {measured_rmse[0]:.4f}", f"Scan with ACDC\nRMSE = {measured_rmse[1]:.4f}"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), layout="constrained", sharex=True, sharey=True)
    for ax, image, title in zip(axes, measured_comparison_images, titles):
        plot = ax.imshow(image, cmap="gray", vmin=0, vmax=comparison_max, origin="lower",
                         extent=comparison_extent, interpolation="nearest", aspect="equal")
        ax.set(title=title, xlabel="X: readout [mm]", ylabel="Y: phase [mm]")
    fig.colorbar(plot, ax=axes, label=f"Magnitude [{rmse_units}]")
    show_figure(fig)

## Interprétation des sous-spins

Les sous-spins sont présents dans chaque calcul du signal et de son gradient,
pas uniquement dans les figures finales. Une différence de B0 entre enfants
produit des phases différentes avant la somme complexe. La reconstruction et
la cible idéale utilisent la même discrétisation dans chaque version.

La V2 ne connaît pas la variation mesurée de B0 dans l'épaisseur. La V3 exploite
la variation dans le plan résolue par le B0 natif, mais l'amplitude GRE reste une
mesure plus grossière interpolée : elle n'apporte pas de nouveaux détails tissulaires.
Le recalage physique, le signe B0 et la rotation hérités du 04 sont conservés.
Le profil RF réel, le B0 volumique et les mouvements restent des extensions distinctes.